# Brain Tumour Subtype Classification: Unified Benchmark of Nine Backbones

Three-way classification of tumour-positive brain MRI slices into glioma, meningioma and pituitary tumour with nine ImageNet-pretrained backbones: classical CNNs (ResNet50, VGG16, DenseNet121), efficient CNNs (EfficientNetB0, MobileNetV3-Large, InceptionV3), and transformer or hybrid models (ViT-B/16, Swin-Base, CoAtNet-0). The detection notebook runs the identical pipeline on tumour presence; subtype classification is only meaningful after a tumour-positive decision.

This one notebook replaces the per-architecture notebooks. Every backbone is trained and evaluated by the same code, on the same splits, with the same optimisation budget, so the only thing that differs between rows of the results tables is the architecture.

**What changed relative to the earlier notebooks and the paper's stated limitations**

| Limitation or pending requirement | How this notebook handles it |
| --- | --- |
| Binary pool not deduplicated before splitting (82.6% of test images had a training near-duplicate) | The identical perceptual-hash pass runs on both tasks **before** any split |
| Deduplication order-dependent across labels; cross-label near-duplicates neither counted nor handled | Label-aware deduplication; cross-label near-identical pairs (NCC ≥ 0.98) are label noise and both copies are excluded; all cross-label pairs are counted and shown |
| Slice-level split; correlated slices can straddle the partition | Group-aware split: images linked by Hamming ≤ `GROUP_TOLERANCE` or NCC ≥ 0.98 form one group and never cross train, validation and test; a leakage audit asserts this |
| Single run per model, no confidence intervals | 5-fold cross-validation on the development pool plus a fixed held-out test set, fold-ensemble predictions, bootstrap 95% CIs, exact McNemar tests with Holm correction |
| Optimisation budget confounded with batch size (80 to 350 steps) | Every model uses the same effective batch (gradient accumulation) and therefore the same number of optimiser steps |
| Swin-Base from different libraries per task; InceptionV3 auxiliary loss, single-GPU run and unused head; MobileNetV3 pinned to old weights | One model registry for both tasks; auxiliary head removed for every task; every model on the library default weights |
| Only 6 of 9 backbones run on subtype classification | All 9 backbones run on both tasks |
| Poor calibration and badly placed thresholds (MobileNetV3: Youden threshold 0.00044) | Temperature scaling and the binary threshold are fitted on each validation fold and applied to the test set |
| Checkpoint chosen on a saturated AUC (ties at 1.0000) | Checkpoint chosen on validation log-loss, a continuous proper scoring rule |
| Outside-head attention measured only for EfficientNetB0; Grad-CAM only qualitative for transformers; VGG16 target was a pooling layer | Outside-head energy for every model on every test image; attention rollout for ViT; model-agnostic occlusion maps for all; VGG16 targets its last convolution |
| Background-only retraining not performed | Each model is retrained on background-only images (head masked) and tested against the majority-class rate |
| Latency without variance; training time not comparable | Latency and throughput repeated three times with standard deviations; training time, throughput and peak memory per run |
| Training images uncropped while the web app crops to the brain | Training and evaluation use the web app's exact crop (`crop_to_brain`), which also removes most source-specific framing |
| No external validation | Optional `EXTERNAL_FOLDERS` hook: any attached collection is near-duplicate-filtered against the pool and evaluated |
| `drop_last=True` discarded images every epoch | Every training image is used every epoch |

What cannot be fixed in a notebook remains a limitation and is restated at the end: the MRI sequence is unknown, the negative class is tumour-negative rather than certified normal, patient identifiers are unavailable (the group split is a proxy), and no institutional three-dimensional data are used.

**Data.** Bhuvaji et al., *Brain Tumor Classification (MRI)* (`sartajbhuvaji/brain-tumor-classification-mri`, training and testing folders merged, its tumour-negative folder excluded), and the Cheng et al. figshare collection of contrast-enhanced T1 slices in the JPEG conversion `moundir8/jun-cheng-brain-tumor-mri-dataset-jpg`. The Bhuvaji collection records no MRI sequence, so the pooled set is a mixture of unknown sequence composition.

**Hardware.** Kaggle GPU T4 x2. Two worker processes train different (model, fold) jobs in parallel, one per GPU; augmentation runs on the GPU so the four CPU cores only feed cached tensors.

**Outputs in `/kaggle/working/`**

```text
weights/<model>/<run>.pth   best-validation checkpoint of every run
weights/deploy/             EfficientNetB0 checkpoint for the web app + metadata.json
metrics/                    tables (CSV) and summaries (JSON); metrics/runs/<run>/ per-run history and summary
predictions/<run>.npz       validation, test, masked-test and external logits per run
plots/                      every figure as PNG
logs/                       notebook log, per-GPU worker logs and epoch events
src/                        the library and worker written by this notebook
outputs.zip                 everything above, built by the last cell
```

**Required Kaggle settings.** Accelerator **GPU T4 x2**; **Internet on** (pretrained weights are downloaded); inputs: `sartajbhuvaji/brain-tumor-classification-mri` and `moundir8/jun-cheng-brain-tumor-mri-dataset-jpg`. To continue an interrupted run, attach the previous version's output and set `RESUME_DIR`.

**Runtime.** Not measured yet: the notebook has not been run in this form. The training stage stops claiming new jobs after `TRAINING_BUDGET_HOURS`, so the analysis always runs on whatever finished.

## 1. Setup

Imports, plus `imagehash` (perceptual hashing) and `timm` (CoAtNet-0 only), which are installed on demand if the Kaggle image lacks them. Expected output: the library versions.

In [ ]:
import importlib
import subprocess
import sys


def ensure(module_name, pip_name=None):
    try:
        return importlib.import_module(module_name)
    except ImportError:
        print(f"Installing {pip_name or module_name} (requires Internet)...", flush=True)
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or module_name]
        )
        return importlib.import_module(module_name)


imagehash = ensure("imagehash", "ImageHash")
timm = ensure("timm")

import json
import math
import os
import shutil
import time
import warnings
import zipfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import FileLink, display
from PIL import Image
from scipy.stats import binomtest
from sklearn import metrics as skm
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.flop_counter")
print(
    f"torch {torch.__version__} | torchvision {torchvision.__version__} | timm {timm.__version__} | "
    f"numpy {np.__version__} | pandas {pd.__version__}",
    flush=True,
)

### Configuration

Every path, seed and hyperparameter lives here. Key choices:

- `EFFECTIVE_BATCH` and `EPOCHS` fix the number of optimiser steps for **every** model; each model's micro-batch is probed to fit its GPU and gradient accumulation makes up the difference.
- `N_FOLDS` folds of the development pool plus one held-out test set of about `TEST_FRACTION` of the images, all drawn by group.
- `DEDUP_TOLERANCE = 5` reproduces the paper's pass; `GROUP_TOLERANCE` links visually similar images (for example adjacent slices) into one split group.
- `SHORTCUT_TEST` adds one background-only training run per model.
- `TRAINING_BUDGET_HOURS` bounds the training stage inside Kaggle's 12-hour limit.

In [ ]:
TASK = "multiclass"
CLASS_NAMES = ["Glioma", "Meningioma", "Pituitary"]
# label -> [(dataset slug, folder inside the dataset, subset name)]; the subset's first word is the source.
# Folder order fixes which copy of a duplicate group is kept.
BHUVAJI, CHENG = "brain-tumor-classification-mri", "jun-cheng-brain-tumor-mri-dataset-jpg"
FOLDERS = {
    0: [
        (BHUVAJI, "Testing/glioma_tumor", "Bhuvaji testing"),
        (BHUVAJI, "Training/glioma_tumor", "Bhuvaji training"),
        (CHENG, "brain tumor mri dataset/2_glioma", "Cheng"),
    ],
    1: [
        (BHUVAJI, "Testing/meningioma_tumor", "Bhuvaji testing"),
        (BHUVAJI, "Training/meningioma_tumor", "Bhuvaji training"),
        (CHENG, "brain tumor mri dataset/1_meningioma", "Cheng"),
    ],
    2: [
        (BHUVAJI, "Testing/pituitary_tumor", "Bhuvaji testing"),
        (BHUVAJI, "Training/pituitary_tumor", "Bhuvaji training"),
        (CHENG, "brain tumor mri dataset/3_pituitary_tumor", "Cheng"),
    ],
}
EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")
# Optional external test collection in the same format, never used for training or selection.
EXTERNAL_FOLDERS = {}

# Shared settings for both 2D tasks
MODELS = [
    "ResNet50",
    "VGG16",
    "DenseNet121",
    "EfficientNetB0",
    "MobileNetV3-Large",
    "InceptionV3",
    "ViT-B/16",
    "Swin-Base",
    "CoAtNet-0",
]
SEED = 42
N_FOLDS = 5
TEST_FRACTION = 0.15
EPOCHS = 20
WARMUP_EPOCHS = 1
EFFECTIVE_BATCH = 64
LR = 3e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
DETERMINISTIC = False  # True trades speed for bit-reproducible kernels
CACHE_SIZE = 320  # side of the cached brain crops; covers InceptionV3's 299 px input
DEDUP_TOLERANCE = 5
GROUP_TOLERANCE = 8
NCC_DUPLICATE = 0.98
SHORTCUT_TEST = True
BOOTSTRAP_SAMPLES = 2000
GRADCAM_BATCH = 16
OCCLUSION_IMAGES_PER_CLASS = 8
OCCLUSION_PATCH, OCCLUSION_STRIDE = 32, 16
LATENCY_WARMUP, LATENCY_ITERS = 10, 100
THROUGHPUT_BATCH, THROUGHPUT_WARMUP, THROUGHPUT_ITERS = 32, 3, 20
BENCH_REPEATS = 3
TRAINING_BUDGET_HOURS = 9.0
RESUME_DIR = None  # e.g. Path("/kaggle/input/<previous-version-output>") to continue a run
DEPLOY_MODEL = "EfficientNetB0"  # architecture loaded by the web app

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SRC, CACHE = WORK / "src", WORK / "cache"
PLOTS, METRICS, LOGS = WORK / "plots", WORK / "metrics", WORK / "logs"
WEIGHTS, PREDICTIONS = WORK / "weights", WORK / "predictions"
RUNS, CLAIMS = METRICS / "runs", WORK / "cache" / "claims"
for folder in (SRC, CACHE, PLOTS, METRICS, LOGS, WEIGHTS, PREDICTIONS, RUNS, CLAIMS):
    folder.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = len(CLASS_NAMES)
NUM_OUTPUTS = 1 if NUM_CLASSES == 2 else NUM_CLASSES
CLASS_COLOURS = plt.get_cmap("tab10").colors[:NUM_CLASSES]
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "axes.grid": True, "grid.alpha": 0.3})
np.random.seed(SEED)
torch.manual_seed(SEED)
NOTEBOOK_START = time.time()


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(LOGS / "notebook.log", "a") as handle:
        handle.write(line + "\n")


def save_fig(fig, name):
    path = PLOTS / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    log(f"Saved plot {path.name}")


def save_table(frame, name, index=False, show=True):
    frame.to_csv(METRICS / f"{name}.csv", index=index)
    if show:
        display(frame)


def save_json(data, name):
    (METRICS / f"{name}.json").write_text(json.dumps(data, indent=2, default=float))


log(f"Task: {TASK} | classes: {CLASS_NAMES} | models: {len(MODELS)} | folds: {N_FOLDS}")

### Hardware check

Prints the GPUs, CPU cores, RAM and free disk, and stops with a clear message unless two GPUs are visible. Expected output: two Tesla T4 lines.

In [ ]:
GPU_COUNT = torch.cuda.device_count()
for index in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(index)
    log(f"GPU {index}: {props.name}, {props.total_memory / 1024**3:.1f} GB")
memory = psutil.virtual_memory()
log(
    f"CPU cores: {os.cpu_count()} | RAM {memory.total / 1e9:.1f} GB total, {memory.available / 1e9:.1f} GB available"
)
log(f"Free disk in /kaggle/working: {shutil.disk_usage(WORK).free / 1e9:.1f} GB")
assert (
    GPU_COUNT >= 2
), "Two GPUs are not visible: select 'GPU T4 x2' as the accelerator in the notebook settings."
DEVICE = torch.device("cuda:0")

### Shared library

Written to `src/image_lib.py` so the notebook and both GPU workers import identical code. It holds the web app's brain crop, the head mask (Otsu, closing, largest component, hole filling), perceptual hashing and the duplicate tools, the model registry with each backbone's Grad-CAM layer, GPU augmentation, calibration and statistics, Grad-CAM, attention rollout, occlusion maps, timing, and the live worker monitor.

In [ ]:
%%writefile /kaggle/working/src/image_lib.py
"""Shared code for the unified brain tumour detection and classification notebooks.

The notebook writes this file to /kaggle/working/src so that the notebook and the
training workers (one process per GPU) import exactly the same definitions.
"""

from __future__ import annotations

import io
import math
import re
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
THUMB_SIZE = 64  # thumbnail side used by the normalised cross-correlation duplicate detector

# Every backbone is built from one registry for both tasks, so a model name always means the
# same implementation and the same pretrained checkpoint.
MODEL_SPECS = {
    "ResNet50": {"family": "Classical CNN", "input_size": 224, "micro_batch": 64},
    "VGG16": {"family": "Classical CNN", "input_size": 224, "micro_batch": 64},
    "DenseNet121": {"family": "Classical CNN", "input_size": 224, "micro_batch": 64},
    "EfficientNetB0": {"family": "Efficient CNN", "input_size": 224, "micro_batch": 64},
    "MobileNetV3-Large": {"family": "Efficient CNN", "input_size": 224, "micro_batch": 64},
    "InceptionV3": {"family": "Efficient CNN", "input_size": 299, "micro_batch": 64},
    "ViT-B/16": {"family": "Transformer", "input_size": 224, "micro_batch": 64},
    "Swin-Base": {"family": "Transformer", "input_size": 224, "micro_batch": 32},
    "CoAtNet-0": {"family": "Hybrid", "input_size": 224, "micro_batch": 64},
}
TOKEN_MODELS = {"ViT-B/16", "Swin-Base"}  # no channels_last benefit for these


def slug(name):
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def run_id(model, fold, mode):
    return f"{slug(model)}_fold{fold}" + ("_background" if mode == "background" else "")


# ---------------------------------------------------------------------------
# Image loading, cropping, hashing
# ---------------------------------------------------------------------------


def crop_to_brain(bgr):
    """The crop the web app applies before inference (Backend/neuroscan/inference/processor.py).

    Training on the same crop removes the train/serve mismatch of the earlier notebooks and
    removes most of the border and framing that differs between source collections.
    """
    grey = cv2.GaussianBlur(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, thresh = cv2.threshold(grey, 45, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        bgr = bgr[y : y + h, x : x + w]
    return bgr


def head_mask(gray):
    """Otsu threshold, closing, largest component, hole filling (as in the revision audit)."""
    from scipy import ndimage
    from skimage.filters import threshold_otsu

    try:
        foreground = gray > threshold_otsu(gray)
    except ValueError:
        return np.ones_like(gray, dtype=bool)
    foreground = ndimage.binary_closing(foreground, iterations=2)
    labels, count = ndimage.label(foreground)
    if count == 0:
        return np.ones_like(gray, dtype=bool)
    sizes = ndimage.sum(foreground, labels, range(1, count + 1))
    return ndimage.binary_fill_holes(labels == int(np.argmax(sizes)) + 1)


def describe_image(args):
    """Decode one file once and return everything later stages need.

    The perceptual hash and thumbnail are computed on the original image (as in the paper);
    the cached training image is the brain crop resized to cache_size x cache_size RGB.
    """
    import imagehash
    from PIL import Image

    path, cache_size = args
    empty = np.zeros((cache_size, cache_size, 3), np.uint8)
    try:
        data = Path(path).read_bytes()
        with Image.open(io.BytesIO(data)) as img:
            width, height = img.size
            rgb = img.convert("RGB")
        bits = np.packbits(imagehash.phash(rgb).hash.flatten().astype(np.uint8))
        hash_value = int.from_bytes(bits.tobytes(), "big")
        thumb = np.asarray(
            rgb.convert("L").resize((THUMB_SIZE, THUMB_SIZE), Image.BILINEAR), dtype=np.uint8
        )
        bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        if bgr is None:  # formats OpenCV cannot decode: fall back to the PIL decode
            bgr = cv2.cvtColor(np.asarray(rgb), cv2.COLOR_RGB2BGR)
        cropped = crop_to_brain(bgr)
        crop_fraction = cropped.shape[0] * cropped.shape[1] / (bgr.shape[0] * bgr.shape[1])
        cached = cv2.cvtColor(
            cv2.resize(cropped, (cache_size, cache_size), interpolation=cv2.INTER_AREA),
            cv2.COLOR_BGR2RGB,
        )
        mask = head_mask(cv2.cvtColor(cached, cv2.COLOR_RGB2GRAY))
        return {
            "ok": True,
            "width": width,
            "height": height,
            "hash": hash_value,
            "thumb": thumb,
            "image": cached,
            "mask": mask.astype(np.uint8),
            "crop_fraction": crop_fraction,
        }
    except Exception as exc:  # unreadable files are reported and excluded, never silently kept
        print(f"Unreadable image {path}: {exc}", flush=True)
        return {
            "ok": False,
            "width": 0,
            "height": 0,
            "hash": 0,
            "thumb": np.zeros((THUMB_SIZE, THUMB_SIZE), np.uint8),
            "image": empty,
            "mask": np.zeros((cache_size, cache_size), np.uint8),
            "crop_fraction": 0.0,
        }


_POPCOUNT = np.array([bin(value).count("1") for value in range(256)], dtype=np.uint8)


def popcount64(values):
    values = np.ascontiguousarray(values, dtype=np.uint64)
    return _POPCOUNT[values.view(np.uint8)].reshape(*values.shape, 8).sum(axis=-1, dtype=np.uint8)


def hamming_matrix(hashes_a, hashes_b, chunk=512):
    """Pairwise Hamming distances between two lists of 64-bit hashes."""
    hashes_a = np.asarray(hashes_a, dtype=np.uint64)
    hashes_b = np.asarray(hashes_b, dtype=np.uint64)
    out = np.empty((len(hashes_a), len(hashes_b)), dtype=np.uint8)
    for start in range(0, len(hashes_a), chunk):
        out[start : start + chunk] = popcount64(
            hashes_a[start : start + chunk, None] ^ hashes_b[None, :]
        )
    return out


def ncc_vectors(thumbs):
    vectors = thumbs.reshape(len(thumbs), -1).astype(np.float32)
    vectors -= vectors.mean(axis=1, keepdims=True)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-8
    return vectors


def greedy_dedup(distance, valid, tolerance, labels):
    """Label-aware greedy deduplication in a fixed order.

    An image is dropped when a previously kept image of the same label lies within the
    tolerance, so the outcome no longer depends on the order in which classes are read.
    """
    kept_by_label = {}
    keep = np.zeros(len(distance), dtype=bool)
    nearest = np.full(len(distance), -1)
    for index in range(len(distance)):
        if not valid[index]:
            continue
        pool = kept_by_label.setdefault(labels[index], [])
        if pool:
            row = distance[index, pool]
            best = int(row.argmin())
            if row[best] <= tolerance:
                nearest[index] = pool[best]
                continue
        pool.append(index)
        keep[index] = True
    return keep, nearest


def connected_groups(adjacency_pairs, count):
    """Union-find over index pairs; returns a group id per index."""
    parent = np.arange(count)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for a, b in adjacency_pairs:
        root_a, root_b = find(a), find(b)
        if root_a != root_b:
            parent[max(root_a, root_b)] = min(root_a, root_b)
    return np.array([find(i) for i in range(count)])


# ---------------------------------------------------------------------------
# Models
# ---------------------------------------------------------------------------


def build_model(name, num_outputs, pretrained=True):
    tv = torchvision.models
    if name == "ResNet50":
        model = tv.resnet50(weights=tv.ResNet50_Weights.DEFAULT if pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_outputs)
    elif name == "VGG16":
        model = tv.vgg16(weights=tv.VGG16_Weights.DEFAULT if pretrained else None)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_outputs)
    elif name == "DenseNet121":
        model = tv.densenet121(weights=tv.DenseNet121_Weights.DEFAULT if pretrained else None)
        model.classifier = nn.Linear(model.classifier.in_features, num_outputs)
    elif name == "EfficientNetB0":
        model = tv.efficientnet_b0(weights=tv.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_outputs)
    elif name == "MobileNetV3-Large":
        model = tv.mobilenet_v3_large(
            weights=tv.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        )
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_outputs)
    elif name == "InceptionV3":
        # transform_input is fixed so a skeleton built without weights matches the trained model.
        model = tv.inception_v3(
            weights=tv.Inception_V3_Weights.DEFAULT if pretrained else None,
            transform_input=True,
            init_weights=False,
        )
        # The auxiliary classifier is removed for every task, so no model receives an extra
        # loss term and no unused parameters are counted.
        model.aux_logits = False
        model.AuxLogits = None
        model.fc = nn.Linear(model.fc.in_features, num_outputs)
    elif name == "ViT-B/16":
        model = tv.vit_b_16(weights=tv.ViT_B_16_Weights.DEFAULT if pretrained else None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_outputs)
    elif name == "Swin-Base":
        model = tv.swin_b(weights=tv.Swin_B_Weights.DEFAULT if pretrained else None)
        model.head = nn.Linear(model.head.in_features, num_outputs)
    elif name == "CoAtNet-0":
        import timm  # CoAtNet has no torchvision implementation

        model = timm.create_model("coatnet_0_rw_224", pretrained=pretrained, num_classes=num_outputs)
    else:
        raise KeyError(f"Unknown model {name!r}; choose from {list(MODEL_SPECS)}")
    return model


def gradcam_layer(model, name):
    """Last spatial or token stage of each backbone."""
    if name == "ResNet50":
        return model.layer4[-1]
    if name == "VGG16":
        return model.features[28]  # last convolution, not the final max-pooling layer
    if name == "DenseNet121":
        return model.features.norm5
    if name in ("EfficientNetB0", "MobileNetV3-Large"):
        return model.features[-1]
    if name == "InceptionV3":
        return model.Mixed_7c
    if name == "ViT-B/16":
        return model.encoder.layers[-1].ln_1
    if name == "Swin-Base":
        return model.norm  # final norm, (B, 7, 7, C) channels-last grid
    if name == "CoAtNet-0":
        return model.stages[-1]
    raise KeyError(name)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def load_weights(model, path, device):
    state = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict({key.removeprefix("module."): value for key, value in state.items()})
    return model


# ---------------------------------------------------------------------------
# GPU preprocessing and augmentation
# ---------------------------------------------------------------------------


class GpuPreprocessor:
    """Resize, augment and normalise uint8 batches on the GPU.

    Augmentation matches the earlier notebooks (horizontal flip p=0.5, rotation up to 10
    degrees, brightness and contrast jitter of 20 percent) but runs on the GPU, so the four
    Kaggle CPU cores are not the bottleneck. ``mode`` selects the input variant:
    "full", "background" (head region set to zero) or "head" (background set to zero).
    """

    def __init__(self, input_size, device, channels_last=False):
        self.size = input_size
        self.device = device
        self.channels_last = channels_last
        self.mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)

    def __call__(self, images, masks=None, mode="full", train=False):
        x = images.to(self.device, non_blocking=True).permute(0, 3, 1, 2).float().div_(255)
        if mode != "full":
            head = masks.to(self.device, non_blocking=True).float().unsqueeze(1)
            x = x * (1 - head) if mode == "background" else x * head
        x = F.interpolate(x, size=(self.size, self.size), mode="bilinear", antialias=True)
        if train:
            batch = x.shape[0]
            flip = torch.rand(batch, device=self.device) < 0.5
            x = torch.where(flip[:, None, None, None], x.flip(-1), x)
            angle = (torch.rand(batch, device=self.device) * 2 - 1) * math.radians(10)
            cos, sin = torch.cos(angle), torch.sin(angle)
            zeros = torch.zeros_like(cos)
            theta = torch.stack(
                [torch.stack([cos, -sin, zeros], 1), torch.stack([sin, cos, zeros], 1)], 1
            )
            grid = F.affine_grid(theta, list(x.shape), align_corners=False)
            x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
            brightness = torch.empty(batch, 1, 1, 1, device=self.device).uniform_(0.8, 1.2)
            x = (x * brightness).clamp_(0, 1)
            contrast = torch.empty(batch, 1, 1, 1, device=self.device).uniform_(0.8, 1.2)
            grey_mean = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).mean(
                dim=(1, 2)
            ).view(-1, 1, 1, 1)
            x = ((x - grey_mean) * contrast + grey_mean).clamp_(0, 1)
        x = (x - self.mean) / self.std
        if self.channels_last:
            x = x.contiguous(memory_format=torch.channels_last)
        return x


def denormalise(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.detach().float().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


# ---------------------------------------------------------------------------
# Inference helpers
# ---------------------------------------------------------------------------


@torch.no_grad()
def predict_logits(model, prep, images, masks=None, mode="full", batch_size=128):
    """Logits for a uint8 array (N, S, S, 3); masks are needed for the masked modes."""
    model.eval()
    outputs = []
    for start in range(0, len(images), batch_size):
        batch = torch.from_numpy(np.ascontiguousarray(images[start : start + batch_size]))
        batch_masks = (
            None
            if masks is None
            else torch.from_numpy(np.ascontiguousarray(masks[start : start + batch_size]))
        )
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(prep(batch, batch_masks, mode=mode))
        outputs.append(logits.float().cpu())
    return torch.cat(outputs).numpy()


def logits_to_probs(logits, temperature=1.0):
    """Class probabilities (N, K); a single-logit binary model gives two columns."""
    logits = np.asarray(logits, dtype=np.float64) / temperature
    if logits.ndim == 1 or logits.shape[1] == 1:
        positive = 1 / (1 + np.exp(-logits.reshape(-1)))
        return np.stack([1 - positive, positive], axis=1)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def fit_temperature(logits, labels):
    """Temperature that minimises validation negative log-likelihood (Guo et al., 2017)."""
    from scipy.optimize import minimize_scalar

    labels = np.asarray(labels, dtype=int)

    def nll(log_t):
        probs = logits_to_probs(logits, math.exp(log_t))
        return -np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1)))

    result = minimize_scalar(nll, bounds=(-3, 3), method="bounded")
    return float(math.exp(result.x))


# ---------------------------------------------------------------------------
# Metrics and statistics
# ---------------------------------------------------------------------------


def calibration_table(y_true, probs, n_bins=15):
    import pandas as pd

    confidence = probs.max(axis=1)
    correct = probs.argmax(axis=1) == y_true
    edges = np.linspace(0, 1, n_bins + 1)
    rows = []
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (confidence > low) & (confidence <= high)
        rows.append(
            {
                "bin_low": low,
                "bin_high": high,
                "count": int(mask.sum()),
                "mean_confidence": float(confidence[mask].mean()) if mask.any() else np.nan,
                "accuracy": float(correct[mask].mean()) if mask.any() else np.nan,
            }
        )
    return pd.DataFrame(rows)


def expected_calibration_error(y_true, probs, n_bins=15):
    table = calibration_table(y_true, probs, n_bins).dropna()
    weights = table["count"] / len(y_true)
    return float((weights * (table["accuracy"] - table["mean_confidence"]).abs()).sum())


def predictions_from_probs(probs, threshold=None):
    """Arg-max, or for binary tasks the positive-class threshold when one is given."""
    if probs.shape[1] == 2 and threshold is not None:
        return (probs[:, 1] >= threshold).astype(int)
    return probs.argmax(axis=1)


def per_class_metrics(y_true, y_pred, probs, class_names):
    import pandas as pd
    from sklearn import metrics as skm

    k = len(class_names)
    cm = skm.confusion_matrix(y_true, y_pred, labels=list(range(k)))
    rows = []
    for index, name in enumerate(class_names):
        tp = cm[index, index]
        fn = cm[index].sum() - tp
        fp = cm[:, index].sum() - tp
        tn = cm.sum() - tp - fn - fp
        positive = (y_true == index).astype(int)
        rows.append(
            {
                "class": name,
                "support": int(cm[index].sum()),
                "tp": int(tp),
                "fp": int(fp),
                "fn": int(fn),
                "tn": int(tn),
                "precision": tp / (tp + fp) if tp + fp else 0.0,
                "sensitivity": tp / (tp + fn) if tp + fn else 0.0,
                "specificity": tn / (tn + fp) if tn + fp else 0.0,
                "f1": 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0,
                "roc_auc": safe_score(skm.roc_auc_score, positive, probs[:, index]),
                "average_precision": safe_score(
                    skm.average_precision_score, positive, probs[:, index]
                ),
            }
        )
    return pd.DataFrame(rows)


def safe_score(fn, *args, **kwargs):
    try:
        return float(fn(*args, **kwargs))
    except ValueError:
        return float("nan")


def classification_metrics(y_true, probs, threshold=None):
    """Headline metrics of one prediction set; ``threshold`` applies to binary tasks only."""
    from sklearn import metrics as skm

    y_true = np.asarray(y_true, dtype=int)
    k = probs.shape[1]
    y_pred = predictions_from_probs(probs, threshold)
    labels = list(range(k))
    result = {
        "n": int(len(y_true)),
        "accuracy": skm.accuracy_score(y_true, y_pred),
        "balanced_accuracy": skm.balanced_accuracy_score(y_true, y_pred),
        "precision_macro": skm.precision_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
        "recall_macro": skm.recall_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
        "f1_macro": skm.f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "mcc": skm.matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": skm.cohen_kappa_score(y_true, y_pred),
        "log_loss": skm.log_loss(y_true, np.clip(probs, 1e-12, 1), labels=labels),
        "brier": float(np.mean(np.sum((probs - np.eye(k)[y_true]) ** 2, axis=1))),
        "ece": expected_calibration_error(y_true, probs),
    }
    if k == 2:
        cm = skm.confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        result.update(
            {
                "sensitivity": tp / (tp + fn) if tp + fn else 0.0,
                "specificity": tn / (tn + fp) if tn + fp else 0.0,
                "false_negatives": int(fn),
                "false_positives": int(fp),
                "roc_auc": safe_score(skm.roc_auc_score, y_true, probs[:, 1]),
                "pr_auc": safe_score(skm.average_precision_score, y_true, probs[:, 1]),
            }
        )
    else:
        result.update(
            {
                "roc_auc": safe_score(
                    skm.roc_auc_score, y_true, probs, multi_class="ovr", average="macro"
                ),
                "pr_auc": safe_score(
                    skm.average_precision_score, np.eye(k)[y_true], probs, average="macro"
                ),
            }
        )
    return {key: float(value) for key, value in result.items()}


def youden_threshold(y_true, positive_probs):
    from sklearn import metrics as skm

    fpr, tpr, thresholds = skm.roc_curve(y_true, positive_probs)
    best = int(np.argmax(tpr - fpr))
    return float(min(thresholds[best], 1.0))


def bootstrap_ci(y_true, probs, metric_fn, n_boot=2000, seed=0, threshold=None):
    """Percentile 95 percent interval of a metric over test-set resamples."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    values = []
    for _ in range(n_boot):
        index = rng.integers(0, n, n)
        if len(np.unique(y_true[index])) < probs.shape[1]:
            continue  # a resample missing a class has no defined AUC
        values.append(metric_fn(y_true[index], probs[index], threshold))
    return float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))


def mcnemar_p(correct_a, correct_b):
    """Exact McNemar test on paired correctness vectors."""
    from scipy.stats import binomtest

    only_a = int(np.sum(correct_a & ~correct_b))
    only_b = int(np.sum(~correct_a & correct_b))
    if only_a + only_b == 0:
        return 1.0
    return float(binomtest(only_a, only_a + only_b, 0.5).pvalue)


# ---------------------------------------------------------------------------
# Attention maps
# ---------------------------------------------------------------------------


def to_nchw(tensor):
    if tensor.dim() == 3:  # transformer tokens (batch, tokens, channels)
        tokens = tensor.shape[1]
        side = int(round(math.sqrt(tokens)))
        if side * side != tokens:  # drop the class token
            tensor = tensor[:, 1:, :]
            side = int(round(math.sqrt(tensor.shape[1])))
        return tensor.reshape(tensor.shape[0], side, side, tensor.shape[2]).permute(0, 3, 1, 2)
    if tensor.dim() == 4 and tensor.shape[1] == tensor.shape[2] and tensor.shape[3] != tensor.shape[1]:
        return tensor.permute(0, 3, 1, 2)  # channels-last grid
    return tensor


def normalise_maps(maps):
    maps = maps - maps.flatten(1).min(dim=1).values[:, None, None]
    peak = maps.flatten(1).max(dim=1).values[:, None, None]
    return torch.where(peak > 0, maps / peak.clamp_min(1e-12), maps)


def gradcam_batch(model, layer, inputs, binary):
    """Grad-CAM of the predicted class for a batch (samples are independent in eval mode)."""
    store = {}

    def hook(module, args, output):
        output = output[0] if isinstance(output, (tuple, list)) else output
        store["activation"] = output
        output.register_hook(lambda grad: store.__setitem__("gradient", grad))

    handle = layer.register_forward_hook(hook)
    try:
        model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = model(inputs).float()
            if binary:
                predicted = (logits[:, 0] > 0).long()
                score = torch.where(predicted == 1, logits[:, 0], -logits[:, 0])
            else:
                predicted = logits.argmax(dim=1)
                score = logits.gather(1, predicted[:, None])[:, 0]
            score.sum().backward()
    finally:
        handle.remove()
    activation = to_nchw(store["activation"].detach().float())
    gradient = to_nchw(store["gradient"].detach().float())
    cam = F.relu((gradient.mean(dim=(2, 3), keepdim=True) * activation).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=inputs.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    model.zero_grad(set_to_none=True)
    return normalise_maps(cam), predicted


def vit_rollout(model, inputs):
    """Attention rollout (Abnar and Zuidema, 2020) for the torchvision ViT."""
    attentions, patched = [], []
    for block in model.encoder.layers:
        module, original = block.self_attention, block.self_attention.forward

        def forward(query, key, value, _original=original, **kwargs):
            kwargs["need_weights"], kwargs["average_attn_weights"] = True, False
            output, weights = _original(query, key, value, **kwargs)
            attentions.append(weights.detach().float())
            return output, weights

        module.forward = forward
        patched.append(module)
    try:
        with torch.no_grad():
            model(inputs)
    finally:
        for module in patched:
            del module.forward
    tokens = attentions[0].shape[-1]
    identity = torch.eye(tokens, device=inputs.device)
    rollout = identity.expand(inputs.shape[0], tokens, tokens).clone()
    for attention in attentions:
        attention = 0.5 * attention.mean(dim=1) + 0.5 * identity
        rollout = (attention / attention.sum(dim=-1, keepdim=True)) @ rollout
    side = int(math.sqrt(tokens - 1))
    maps = rollout[:, 0, 1:].reshape(-1, 1, side, side)
    maps = F.interpolate(maps, size=inputs.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    return normalise_maps(maps)


@torch.no_grad()
def occlusion_maps(model, inputs, binary, patch=32, stride=32):
    """Drop in predicted-class probability when each patch is replaced by the image mean.

    Model-agnostic, so unlike Grad-CAM it is directly comparable between convolutional and
    attention backbones.
    """
    model.eval()
    with torch.autocast("cuda", dtype=torch.float16):
        base = model(inputs).float()
    if binary:
        predicted = (base[:, 0] > 0).long()
        base_score = torch.sigmoid(torch.where(predicted == 1, base[:, 0], -base[:, 0]))
    else:
        predicted = base.argmax(dim=1)
        base_score = base.softmax(dim=1).gather(1, predicted[:, None])[:, 0]
    size = inputs.shape[-1]
    positions = [(y, x) for y in range(0, size - patch + 1, stride) for x in range(0, size - patch + 1, stride)]
    maps = torch.zeros(inputs.shape[0], size, size, device=inputs.device)
    counts = torch.zeros(size, size, device=inputs.device)
    fill = inputs.mean(dim=(2, 3), keepdim=True)
    for y, x in positions:
        occluded = inputs.clone()
        occluded[:, :, y : y + patch, x : x + patch] = fill
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(occluded).float()
        if binary:
            score = torch.sigmoid(torch.where(predicted == 1, out[:, 0], -out[:, 0]))
        else:
            score = out.softmax(dim=1).gather(1, predicted[:, None])[:, 0]
        maps[:, y : y + patch, x : x + patch] += (base_score - score).clamp_min(0)[:, None, None]
        counts[y : y + patch, x : x + patch] += 1
    return normalise_maps(maps / counts.clamp_min(1)), predicted


def outside_fraction(maps, masks):
    """Share of map energy outside the head mask, per image."""
    total = maps.flatten(1).sum(dim=1)
    outside = (maps * (~masks)).flatten(1).sum(dim=1)
    return torch.where(total > 0, outside / total, torch.full_like(total, float("nan")))


# ---------------------------------------------------------------------------
# Efficiency
# ---------------------------------------------------------------------------


def gflops(model, input_size):
    from torch.utils.flop_counter import FlopCounterMode

    model = model.eval()
    with torch.no_grad(), FlopCounterMode(display=False) as counter:
        model(torch.zeros(1, 3, input_size, input_size))
    return counter.get_total_flops() / 1e9


@torch.no_grad()
def time_passes(model, inputs, warmup, iterations):
    for _ in range(warmup):
        with torch.autocast("cuda", dtype=torch.float16):
            model(inputs)
    torch.cuda.synchronize(inputs.device)
    timings = []
    for _ in range(iterations):
        start = time.perf_counter()
        with torch.autocast("cuda", dtype=torch.float16):
            model(inputs)
        torch.cuda.synchronize(inputs.device)
        timings.append(time.perf_counter() - start)
    return np.array(timings)

### Training worker

Written to `src/image_worker.py` and launched once per GPU in section 5. For each job it probes the largest micro-batch that fits, trains with AdamW, linear warm-up then cosine decay **per optimiser step**, fp16 autocast with gradient scaling and gradient clipping, evaluates the validation fold every epoch, keeps the checkpoint with the lowest validation log-loss, and finally saves validation, test, masked-test and external logits. Progress is written to small status files that the notebook turns into live progress bars.

In [ ]:
%%writefile /kaggle/working/src/image_worker.py
"""Training worker for the unified 2D notebooks: one process per GPU.

Each worker claims jobs (model, fold, input mode) from a shared list, trains, selects the
checkpoint on the validation fold, and writes predictions, history and a summary. A job is
skipped when its summary already exists, so a rerun or a resumed session continues where
the previous one stopped.
"""

import argparse
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path(__file__).resolve().parent))
import image_lib as lib  # noqa: E402
import shared_lib  # noqa: E402


def seed_everything(seed, deterministic):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)


def loss_sum(logits, labels, binary):
    if binary:
        return F.binary_cross_entropy_with_logits(
            logits[:, 0].float(), labels.float(), reduction="sum"
        )
    return F.cross_entropy(logits.float(), labels, reduction="sum")


def probe_micro_batch(model, prep_size, start, binary, num_outputs, device, channels_last):
    """Largest micro-batch whose forward and backward pass fit, halving on out-of-memory.

    Weights and normalisation statistics are restored afterwards, so probing does not
    change the model that is trained.
    """
    state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    model.train()
    candidate = start
    try:
        while candidate >= 1:
            try:
                x = torch.randn(candidate, 3, prep_size, prep_size, device=device)
                if channels_last:
                    x = x.contiguous(memory_format=torch.channels_last)
                y = torch.zeros(candidate, dtype=torch.long, device=device)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = loss_sum(model(x), y, binary)
                loss.backward()
                torch.cuda.synchronize(device)
                return candidate
            except torch.OutOfMemoryError:
                candidate //= 2
            finally:
                model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
        raise RuntimeError("Even a micro-batch of 1 does not fit in GPU memory")
    finally:
        model.load_state_dict(state)


def evaluate(model, prep, images, masks, labels, mode, binary):
    logits = lib.predict_logits(model, prep, images, masks, mode)
    labels_t = torch.from_numpy(labels)
    loss = loss_sum(torch.from_numpy(logits), labels_t, binary).item() / len(labels)
    probs = lib.logits_to_probs(logits)
    accuracy = float((probs.argmax(1) == labels).mean())
    if binary:
        auc = roc_auc_score(labels, probs[:, 1])
    else:
        auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    return logits, loss, accuracy, float(auc)


def run_job(job, cfg, arrays, frame, device, reporter):
    name, fold, mode = job["model"], job["fold"], job["mode"]
    rid = lib.run_id(name, fold, mode)
    binary = cfg["num_outputs"] == 1
    spec = lib.MODEL_SPECS[name]
    seed_everything(cfg["seed"] + 1000 * fold + (1 if mode == "background" else 0), cfg["deterministic"])
    images, masks = arrays

    train_rows = frame[(frame["role"] == "dev") & (frame["fold"] != fold)]
    val_rows = frame[(frame["role"] == "dev") & (frame["fold"] == fold)]
    test_rows = frame[frame["role"] == "test"]
    external_rows = frame[frame["role"] == "external"]

    def take(rows):
        index = rows["cache_index"].to_numpy()
        return images[index], masks[index], rows["label"].to_numpy()

    train_images, train_masks, train_labels = take(train_rows)
    val_images, val_masks, val_labels = take(val_rows)
    test_images, test_masks, test_labels = take(test_rows)
    # Pinned host tensors: each step only copies one micro-batch to the GPU.
    train_images_t = torch.from_numpy(np.ascontiguousarray(train_images)).pin_memory()
    train_masks_t = torch.from_numpy(np.ascontiguousarray(train_masks)).pin_memory()
    train_labels_t = torch.from_numpy(train_labels).long()

    channels_last = name not in lib.TOKEN_MODELS
    model = lib.build_model(name, cfg["num_outputs"], pretrained=True).to(device)
    if channels_last:
        model = model.to(memory_format=torch.channels_last)
    prep = lib.GpuPreprocessor(spec["input_size"], device, channels_last)

    effective = cfg["effective_batch"]
    micro = probe_micro_batch(
        model, spec["input_size"], min(spec["micro_batch"], effective), binary, cfg["num_outputs"], device, channels_last
    )
    steps_per_epoch = math.ceil(len(train_rows) / effective)
    total_steps = cfg["epochs"] * steps_per_epoch
    warmup_steps = cfg["warmup_epochs"] * steps_per_epoch

    def lr_factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda")
    generator = torch.Generator().manual_seed(cfg["seed"] + fold)

    run_dir = Path(cfg["runs_dir"]) / rid
    run_dir.mkdir(parents=True, exist_ok=True)
    weights_path = Path(cfg["weights_dir"]) / lib.slug(name) / f"{rid}.pth"
    weights_path.parent.mkdir(parents=True, exist_ok=True)
    reporter.event(
        f"{rid}: {len(train_rows)} train / {len(val_rows)} val / {len(test_rows)} test, "
        f"micro-batch {micro} x {math.ceil(effective / micro)} accumulation = {effective}, {total_steps} steps"
    )

    torch.cuda.reset_peak_memory_stats(device)
    history, best_loss, best_epoch, step = [], float("inf"), 0, 0
    train_start = time.perf_counter()
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        epoch_start = time.perf_counter()
        seen, loss_total, correct = 0, 0.0, 0
        for chunk in torch.randperm(len(train_rows), generator=generator).split(effective):
            optimizer.zero_grad(set_to_none=True)
            pieces = list(chunk.split(micro))
            if len(pieces) > 1 and len(pieces[-1]) == 1:  # no single-sample batch-norm steps
                pieces[-2] = torch.cat(pieces[-2:])
                pieces.pop()
            for piece in pieces:
                x = prep(train_images_t[piece], train_masks_t[piece], mode=mode, train=True)
                y = train_labels_t[piece].to(device, non_blocking=True)
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(x)
                loss = loss_sum(logits, y, binary)
                scaler.scale(loss / len(chunk)).backward()
                loss_total += loss.item()
                predicted = (logits[:, 0] > 0).long() if binary else logits.argmax(1)
                correct += int((predicted == y).sum())
                seen += len(piece)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step += 1
            if step % 5 == 0 or step == total_steps:
                reporter.status(
                    job=rid,
                    step=step,
                    total=total_steps,
                    postfix={
                        "epoch": epoch,
                        "loss": round(loss_total / seen, 4),
                        "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                        "mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2),
                    },
                )
        train_seconds = time.perf_counter() - epoch_start
        _, val_loss, val_acc, val_auc = evaluate(model, prep, val_images, val_masks, val_labels, mode, binary)
        row = {
            "epoch": epoch,
            "step": step,
            "lr": scheduler.get_last_lr()[0],
            "train_loss": loss_total / seen,
            "train_acc": correct / seen,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_auc": val_auc,
            "epoch_seconds": time.perf_counter() - epoch_start,
            "train_images_per_second": seen / train_seconds,
            "peak_memory_gb": torch.cuda.max_memory_allocated(device) / 1e9,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        improved = val_loss < best_loss
        if improved:
            best_loss, best_epoch = val_loss, epoch
            torch.save({k: v.detach().float().cpu() for k, v in model.state_dict().items()}, weights_path)
        reporter.event(
            f"{rid} epoch {epoch}/{cfg['epochs']}: train loss {row['train_loss']:.4f} acc {row['train_acc']:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} AUC {val_auc:.4f}"
            + (" | best" if improved else "")
        )
    train_seconds = time.perf_counter() - train_start
    peak_train_memory = torch.cuda.max_memory_allocated(device) / 1e9

    # Predictions of the checkpoint selected on validation loss.
    lib.load_weights(model, weights_path, device)
    outputs = {
        "val_index": val_rows["cache_index"].to_numpy(),
        "val_logits": lib.predict_logits(model, prep, val_images, val_masks, mode),
        "test_index": test_rows["cache_index"].to_numpy(),
        "test_logits": lib.predict_logits(model, prep, test_images, test_masks, mode),
    }
    if mode == "full":  # inference-only masking test on the same checkpoint
        outputs["test_logits_background_only"] = lib.predict_logits(
            model, prep, test_images, test_masks, "background"
        )
        outputs["test_logits_head_only"] = lib.predict_logits(model, prep, test_images, test_masks, "head")
        if len(external_rows):
            ext_images, ext_masks, _ = take(external_rows)
            outputs["external_index"] = external_rows["cache_index"].to_numpy()
            outputs["external_logits"] = lib.predict_logits(model, prep, ext_images, ext_masks, mode)
    np.savez_compressed(Path(cfg["predictions_dir"]) / f"{rid}.npz", **outputs)

    summary = {
        "run_id": rid,
        "model": name,
        "family": spec["family"],
        "fold": fold,
        "mode": mode,
        "input_size": spec["input_size"],
        "parameters": lib.count_parameters(model),
        "micro_batch": micro,
        "accumulation": math.ceil(effective / micro),
        "effective_batch": effective,
        "optimizer_steps": step,
        "epochs": cfg["epochs"],
        "best_epoch": best_epoch,
        "best_val_loss": best_loss,
        "train_seconds": train_seconds,
        "train_images_per_second": len(train_rows) * cfg["epochs"] / train_seconds,
        "peak_train_memory_gb": peak_train_memory,
        "train_images": len(train_rows),
        "val_images": len(val_rows),
        "gpu": torch.cuda.get_device_name(device),
    }
    shared_lib.write_json(run_dir / "summary.json", summary)
    reporter.event(f"{rid} done in {train_seconds / 60:.1f} min, best epoch {best_epoch}")
    del model, optimizer, train_images_t, train_masks_t
    torch.cuda.empty_cache()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--gpu", type=int, required=True)
    args = parser.parse_args()
    cfg = json.loads(Path(args.config).read_text())
    device = torch.device("cuda:0")  # CUDA_VISIBLE_DEVICES selects the physical GPU
    torch.cuda.set_device(device)
    reporter = shared_lib.Reporter(cfg["log_dir"], args.gpu)
    frame = pd.read_csv(cfg["split_table"])
    arrays = (np.load(cfg["images_path"], mmap_mode="r"), np.load(cfg["masks_path"], mmap_mode="r"))
    claims = Path(cfg["claims_dir"])
    for job in cfg["jobs"]:
        rid = lib.run_id(job["model"], job["fold"], job["mode"])
        if (Path(cfg["runs_dir"]) / rid / "summary.json").exists():
            continue
        if time.time() > cfg["deadline"]:
            reporter.event(f"Time budget reached; {rid} and later jobs left for the next session")
            break
        try:
            os.close(os.open(claims / rid, os.O_CREAT | os.O_EXCL | os.O_WRONLY))
        except FileExistsError:
            continue  # the other GPU took this job
        run_job(job, cfg, arrays, frame, device, reporter)
    reporter.status(job=None)


if __name__ == "__main__":
    main()

### Worker launcher and monitor

Written to `src/shared_lib.py`: starts one worker per GPU (each sees a single device through `CUDA_VISIBLE_DEVICES`) and polls their status files to draw an overall jobs bar, one step bar per GPU and a line per finished epoch. A worker that exits with an error stops the cell with the end of its log.

In [ ]:
%%writefile /kaggle/working/src/shared_lib.py
"""Helpers shared by the unified notebooks: worker launch and live monitoring, progress
reporting from the workers, JSON files, and the Holm correction.

Each notebook writes this file to /kaggle/working/src; training runs in one subprocess per GPU
and the notebook turns the workers' status files into live progress bars.
"""

import json
import os
import sys
import time
from pathlib import Path

import numpy as np


def read_json(path, default=None):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        return default


def write_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, default=float))
    os.replace(tmp, path)


def launch_workers(script, config_path, log_dir, gpu_count):
    import subprocess

    processes = []
    for gpu in range(gpu_count):
        env = os.environ.copy()
        env.update(
            CUDA_VISIBLE_DEVICES=str(gpu),
            PYTHONUNBUFFERED="1",
            CUBLAS_WORKSPACE_CONFIG=":4096:8",
            OMP_NUM_THREADS=str(max(1, (os.cpu_count() or 2) // gpu_count)),
        )
        log = open(Path(log_dir) / f"worker_gpu{gpu}.log", "a")
        command = [sys.executable, str(script), "--config", str(config_path), "--gpu", str(gpu)]
        processes.append((subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    return processes


def monitor_workers(processes, log_dir, total_jobs, finished_jobs, poll_seconds=3):
    """Live progress: one bar for all jobs, one bar per GPU, epoch lines as they arrive."""
    from tqdm.auto import tqdm

    log_dir = Path(log_dir)
    overall = tqdm(total=total_jobs, desc="Jobs", unit="job", position=0)
    overall.update(finished_jobs())
    bars = [
        tqdm(total=1, desc=f"GPU {gpu}: waiting", unit="step", position=gpu + 1, leave=True)
        for gpu in range(len(processes))
    ]
    offsets = [0] * len(processes)
    current_job = [None] * len(processes)
    while True:
        for gpu, bar in enumerate(bars):
            status = read_json(log_dir / f"status_gpu{gpu}.json", {})
            if status.get("job"):
                if status["job"] != current_job[gpu]:
                    current_job[gpu] = status["job"]
                    bar.reset(total=max(1, status.get("total", 1)))
                    bar.set_description(f"GPU {gpu}: {status['job']}")
                bar.n = min(status.get("step", 0), bar.total)
                bar.set_postfix(status.get("postfix", {}), refresh=False)
                bar.refresh()
            events = log_dir / f"events_gpu{gpu}.jsonl"
            if events.exists():
                with open(events) as handle:
                    handle.seek(offsets[gpu])
                    for line in handle:
                        tqdm.write(json.loads(line)["message"])
                    offsets[gpu] = handle.tell()
        done = finished_jobs()
        overall.update(done - overall.n)
        if all(process.poll() is not None for process, _ in processes):
            break
        time.sleep(poll_seconds)
    for bar in bars:
        bar.close()
    overall.close()
    failures = []
    for gpu, (process, log) in enumerate(processes):
        log.close()
        if process.returncode != 0:
            tail = (log_dir / f"worker_gpu{gpu}.log").read_text().splitlines()[-40:]
            failures.append(f"GPU {gpu} worker exited with {process.returncode}:\n" + "\n".join(tail))
    if failures:
        raise RuntimeError("\n\n".join(failures))


class Reporter:
    """Status file (polled by the notebook) and epoch events written by a worker."""

    def __init__(self, log_dir, gpu):
        self.status_path = Path(log_dir) / f"status_gpu{gpu}.json"
        self.events_path = Path(log_dir) / f"events_gpu{gpu}.jsonl"
        self.gpu = gpu

    def status(self, **fields):
        write_json(self.status_path, fields)

    def event(self, message, **fields):
        with open(self.events_path, "a") as handle:
            handle.write(json.dumps({"time": time.time(), "message": f"[GPU {self.gpu}] {message}", **fields}) + "\n")
        print(message, flush=True)


def holm(p_values):
    """Holm-Bonferroni adjusted p-values, same order as the input."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[index])
        adjusted[index] = min(1.0, running)
    return adjusted

The library is imported from `src/` so every later cell uses the same functions as the workers. Expected output: the number of registered architectures.

In [ ]:
sys.path.insert(0, str(SRC))
import image_lib as lib
import shared_lib

importlib.reload(lib)
importlib.reload(shared_lib)
assert set(MODELS) <= set(lib.MODEL_SPECS), f"Unknown models: {set(MODELS) - set(lib.MODEL_SPECS)}"
log(f"Library loaded: {len(lib.MODEL_SPECS)} registered architectures")

## 2. Data

### Collect the image pool

Finds each attached dataset under `/kaggle/input` (both the old `/kaggle/input/<slug>` and the newer `/kaggle/input/datasets/<owner>/<slug>` layouts) and lists every image with its source, subset and label. Expected output: images per subset and class.

In [ ]:
def find_dataset_root(slug):
    for candidate in [INPUT / slug, *INPUT.glob(f"*/{slug}"), *INPUT.glob(f"*/*/{slug}")]:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Dataset '{slug}' is not attached: add it with 'Add Input' in the notebook sidebar."
    )


def collect(folder_map, origin):
    rows = []
    for label in sorted(folder_map):
        for slug_name, relative, subset in folder_map[label]:
            root = find_dataset_root(slug_name) / relative
            if not root.is_dir():
                raise FileNotFoundError(f"Expected folder {root} in dataset {slug_name}")
            for directory, _, files in sorted(os.walk(root)):
                for name in sorted(files):
                    if name.lower().endswith(EXTENSIONS):
                        rows.append((os.path.join(directory, name), label, subset, origin))
    return rows


rows = collect(FOLDERS, "pool")
if EXTERNAL_FOLDERS:
    rows += collect(EXTERNAL_FOLDERS, "external")
frame = pd.DataFrame(rows, columns=["path", "label", "subset", "origin"])
frame["source"] = frame["subset"].str.split().str[0]
frame["class_name"] = frame["label"].map(dict(enumerate(CLASS_NAMES)))
frame["cache_index"] = np.arange(len(frame))
log(
    f"Pool: {int((frame['origin'] == 'pool').sum())} images | external: {int((frame['origin'] == 'external').sum())}"
)
display(pd.crosstab(frame["subset"], frame["class_name"], margins=True))

### Decode, hash, crop and cache

One parallel pass over every file (all CPU cores) computes the native size, the 64-bit perceptual hash and a 64 x 64 thumbnail of the **original** image (the inputs of the duplicate detectors), and caches the web app's brain crop at `CACHE_SIZE` px with its head mask. The cache is a memory-mapped array in `cache/`, excluded from the output zip. Expected output: a progress bar, then the count of unreadable files (should be 0).

In [ ]:
N_IMAGES = len(frame)
IMAGES_PATH, MASKS_PATH = CACHE / "images.npy", CACHE / "masks.npy"
images_out = np.lib.format.open_memmap(
    IMAGES_PATH, mode="w+", dtype=np.uint8, shape=(N_IMAGES, CACHE_SIZE, CACHE_SIZE, 3)
)
masks_out = np.lib.format.open_memmap(
    MASKS_PATH, mode="w+", dtype=np.uint8, shape=(N_IMAGES, CACHE_SIZE, CACHE_SIZE)
)
HASHES = np.zeros(N_IMAGES, dtype=np.uint64)
THUMBS = np.zeros((N_IMAGES, lib.THUMB_SIZE, lib.THUMB_SIZE), dtype=np.uint8)
details = {"ok": [], "width": [], "height": [], "crop_fraction": []}
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=os.cpu_count()) as pool:
    jobs = [(path, CACHE_SIZE) for path in frame["path"]]
    for index, info in enumerate(
        tqdm(
            pool.map(lib.describe_image, jobs, chunksize=16),
            total=N_IMAGES,
            desc="Decode, hash, crop",
        )
    ):
        images_out[index], masks_out[index] = info["image"], info["mask"]
        HASHES[index], THUMBS[index] = info["hash"], info["thumb"]
        for key in details:
            details[key].append(info[key])
images_out.flush()
masks_out.flush()
del images_out, masks_out
for key, values in details.items():
    frame[key] = values
frame["hash"] = [f"{int(value):016x}" for value in HASHES]
IMAGES = np.load(IMAGES_PATH, mmap_mode="r")
MASKS = np.load(MASKS_PATH, mmap_mode="r")
log(
    f"Cached {N_IMAGES} images in {time.perf_counter() - start:.0f}s "
    f"({(IMAGES.nbytes + MASKS.nbytes) / 1e9:.2f} GB); unreadable: {int((~frame['ok']).sum())}"
)

### Dataset overview

Provenance (images per source and class), native image sizes, the crop the model actually sees, and the head masks used later for the attention audit. Expected output: a provenance table and three figures.

In [ ]:
pool_frame = frame[(frame["origin"] == "pool") & frame["ok"]]
provenance = pd.crosstab(
    pool_frame["subset"], pool_frame["class_name"], margins=True, margins_name="Total"
)
sizes = pool_frame.groupby("subset").apply(
    lambda g: pd.Series(
        {
            "distinct_sizes": g.groupby(["width", "height"]).ngroups,
            "modal_size": "{}x{}".format(*g.groupby(["width", "height"]).size().idxmax()),
            "modal_share": g.groupby(["width", "height"]).size().max() / len(g),
            "median_crop_fraction": g["crop_fraction"].median(),
        }
    )
)
save_table(provenance.join(sizes), "provenance", index=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pd.crosstab(pool_frame["subset"], pool_frame["class_name"]).plot.bar(
    stacked=True, ax=axes[0], color=CLASS_COLOURS
)
axes[0].set_title("Images per subset and class")
axes[0].tick_params(axis="x", rotation=30)
for label, name in enumerate(CLASS_NAMES):
    rows_ = pool_frame[pool_frame["label"] == label]
    axes[1].scatter(
        rows_["width"], rows_["height"], s=6, alpha=0.4, color=CLASS_COLOURS[label], label=name
    )
    axes[2].hist(rows_["crop_fraction"], bins=40, alpha=0.6, color=CLASS_COLOURS[label], label=name)
axes[1].set(title="Native image size", xlabel="Width (px)", ylabel="Height (px)")
axes[2].set(title="Share of the image kept by the brain crop", xlabel="Crop area / image area")
axes[1].legend()
axes[2].legend()
fig.tight_layout()
save_fig(fig, "01_dataset_overview")

rng = np.random.default_rng(SEED)
samples = [
    rng.choice(pool_frame.index[pool_frame["label"] == label], 4, replace=False)
    for label in range(NUM_CLASSES)
]
fig, axes = plt.subplots(NUM_CLASSES, 12, figsize=(26, 2.4 * NUM_CLASSES), squeeze=False)
for row, chosen in enumerate(samples):
    for k, index in enumerate(chosen):
        with Image.open(frame.at[index, "path"]) as img:
            axes[row, 3 * k].imshow(img.convert("RGB"))
        axes[row, 3 * k + 1].imshow(IMAGES[index])
        axes[row, 3 * k + 2].imshow(IMAGES[index])
        axes[row, 3 * k + 2].contour(MASKS[index], levels=[0.5], colors="lime", linewidths=1)
        axes[row, 3 * k].set_title(f"{CLASS_NAMES[row]}\noriginal", fontsize=8)
        axes[row, 3 * k + 1].set_title("brain crop", fontsize=8)
        axes[row, 3 * k + 2].set_title("head mask", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
fig.suptitle(
    "Original image, brain crop used for training, and head mask used by the attention audit"
)
fig.tight_layout()
save_fig(fig, "02_samples_crop_mask")

## 3. Deduplication

### Label-aware perceptual-hash deduplication

The paper's pass (64-bit pHash, Hamming distance ≤ 5) now runs on **this task's pool before splitting**, and per label, so which copy is kept no longer depends on class order. Pairs within the tolerance that carry **different** labels are classified with the thumbnail normalised cross-correlation: NCC ≥ 0.98 means the same image filed under two labels (label noise, both copies excluded); lower NCC means a hash false positive (both kept, since they are different images). Expected output: removal counts by class and source, and the cross-label pair counts.

In [ ]:
POOL = np.flatnonzero(((frame["origin"] == "pool") & frame["ok"]).to_numpy())
pool_labels = frame["label"].to_numpy()[POOL]
pool_sources = frame["source"].to_numpy()[POOL]
DIST = lib.hamming_matrix(HASHES[POOL], HASHES[POOL])
NCC_VEC = lib.ncc_vectors(THUMBS)
keep, nearest = lib.greedy_dedup(DIST, np.ones(len(POOL), bool), DEDUP_TOLERANCE, pool_labels)

upper = np.triu(DIST <= DEDUP_TOLERANCE, 1)
pair_i, pair_j = np.nonzero(upper & (pool_labels[:, None] != pool_labels[None, :]))
pair_ncc = (NCC_VEC[POOL[pair_i]] * NCC_VEC[POOL[pair_j]]).sum(axis=1)
cross_pairs = pd.DataFrame(
    {
        "path_a": frame["path"].to_numpy()[POOL[pair_i]],
        "path_b": frame["path"].to_numpy()[POOL[pair_j]],
        "class_a": [CLASS_NAMES[v] for v in pool_labels[pair_i]],
        "class_b": [CLASS_NAMES[v] for v in pool_labels[pair_j]],
        "source_a": pool_sources[pair_i],
        "source_b": pool_sources[pair_j],
        "hamming": DIST[pair_i, pair_j],
        "ncc": pair_ncc,
    }
)
cross_pairs["kind"] = np.where(
    cross_pairs["ncc"] >= NCC_DUPLICATE, "label noise (excluded)", "hash false positive (kept)"
)
noisy = set(pair_i[pair_ncc >= NCC_DUPLICATE]) | set(pair_j[pair_ncc >= NCC_DUPLICATE])
label_noise = np.zeros(len(POOL), bool)
label_noise[list(noisy)] = True
save_table(cross_pairs, "cross_label_pairs", show=False)

frame["role"] = np.where(frame["ok"], frame["origin"], "unreadable")
frame.loc[POOL[~keep], "role"] = "duplicate"
frame.loc[POOL[label_noise], "role"] = "label noise"
frame["nearest_kept"] = ""
frame.loc[POOL[nearest >= 0], "nearest_kept"] = frame["path"].to_numpy()[
    POOL[nearest[nearest >= 0]]
]

removed = POOL[~keep]
removal = pd.DataFrame(
    {
        "class_name": frame["class_name"].to_numpy()[removed],
        "removed_source": frame["source"].to_numpy()[removed],
        "kept_source": frame["source"].to_numpy()[POOL[nearest[~keep]]],
    }
)
removal["kind"] = np.where(
    removal["removed_source"] == removal["kept_source"], "within-source", "cross-source"
)
dedup_summary = (
    pd.DataFrame(
        {
            "before": pool_frame["class_name"].value_counts(),
            "duplicates_removed": removal["class_name"].value_counts(),
            "label_noise_excluded": frame.loc[
                POOL[label_noise & keep], "class_name"
            ].value_counts(),
        }
    )
    .reindex(CLASS_NAMES)
    .fillna(0)
    .astype(int)
)
dedup_summary["after"] = (
    dedup_summary["before"]
    - dedup_summary["duplicates_removed"]
    - dedup_summary["label_noise_excluded"]
)
dedup_summary.loc["Total"] = dedup_summary.sum()
dedup_summary["removed_pct"] = (100 * (1 - dedup_summary["after"] / dedup_summary["before"])).round(
    2
)
save_table(dedup_summary, "deduplication_summary", index=True)
save_table(
    pd.crosstab([removal["removed_source"], removal["kind"]], removal["class_name"], margins=True),
    "deduplication_by_source",
    index=True,
)
log(
    f"Cross-label pairs within Hamming {DEDUP_TOLERANCE}: {len(cross_pairs)} "
    f"({int((cross_pairs['ncc'] >= NCC_DUPLICATE).sum())} label noise, {int((cross_pairs['ncc'] < NCC_DUPLICATE).sum())} hash false positives)"
)

### Threshold sensitivity

How many images the label-aware pass keeps at each Hamming tolerance, and the class distribution before and after the pass at the chosen tolerance. Expected output: a table and a two-panel figure.

In [ ]:
sweep = []
for tolerance in tqdm(range(0, 13), desc="Tolerance sweep"):
    kept_t, _ = lib.greedy_dedup(DIST, np.ones(len(POOL), bool), tolerance, pool_labels)
    sweep.append(
        {
            "tolerance": tolerance,
            "kept": int(kept_t.sum()),
            **{
                name: int((kept_t & (pool_labels == k)).sum()) for k, name in enumerate(CLASS_NAMES)
            },
        }
    )
sweep = pd.DataFrame(sweep)
save_table(sweep, "deduplication_threshold_sweep")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(sweep["tolerance"], sweep["kept"], marker="o")
axes[0].axvline(DEDUP_TOLERANCE, color="grey", linestyle="--", label=f"chosen ({DEDUP_TOLERANCE})")
axes[0].set(
    title="Images kept by Hamming tolerance",
    xlabel="Hamming tolerance (bits of 64)",
    ylabel="Images kept",
)
axes[0].legend()
x = np.arange(NUM_CLASSES)
axes[1].bar(x - 0.2, dedup_summary.loc[CLASS_NAMES, "before"], 0.4, label="before")
axes[1].bar(x + 0.2, dedup_summary.loc[CLASS_NAMES, "after"], 0.4, label="after")
axes[1].set_xticks(x, CLASS_NAMES)
axes[1].set(title="Class distribution before and after deduplication", ylabel="Images")
axes[1].legend()
fig.tight_layout()
save_fig(fig, "03_deduplication")

### Cross-label pairs

Up to 12 cross-label pairs, the label-noise pairs first, so the excluded images can be checked by eye. Expected output: a gallery, or a note that no such pairs exist.

In [ ]:
gallery = cross_pairs.sort_values("ncc", ascending=False).head(12)
if gallery.empty:
    log("No cross-label pairs within the tolerance.")
else:
    fig, axes = plt.subplots(2, len(gallery), figsize=(2.3 * len(gallery), 5), squeeze=False)
    for col, pair in enumerate(gallery.itertuples()):
        for row, (path, cls) in enumerate(
            [(pair.path_a, pair.class_a), (pair.path_b, pair.class_b)]
        ):
            with Image.open(path) as img:
                axes[row, col].imshow(img.convert("L"), cmap="gray")
            axes[row, col].set_title(
                f"{cls}\nH={pair.hamming} NCC={pair.ncc:.3f}" if row == 0 else cls, fontsize=7
            )
            axes[row, col].axis("off")
    fig.suptitle("Cross-label near-duplicate pairs (top: first image, bottom: its pair)")
    fig.tight_layout()
    save_fig(fig, "04_cross_label_pairs")

## 4. Group-aware split and leakage audit

Kept images linked by Hamming ≤ `GROUP_TOLERANCE` or thumbnail NCC ≥ 0.98 are joined into groups (connected components), a proxy for patient-level grouping since the sources carry no patient identifiers. `StratifiedGroupKFold` then draws a held-out test set (about `TEST_FRACTION`) and `N_FOLDS` development folds, so no group crosses a partition. The audit measures, for every test image, the nearest development image by Hamming distance and NCC and asserts that none is within the grouping criteria. Optional external images are kept only if they are not near any pool image. Expected output: group statistics, split table, audit table and a figure.

In [ ]:
kept_index = POOL[keep & ~label_noise]
vectors = torch.from_numpy(NCC_VEC[kept_index]).to(DEVICE)
ncc_links = []
for start in tqdm(range(0, len(kept_index), 1024), desc="NCC links"):
    block = vectors[start : start + 1024] @ vectors.T
    hit = (block >= NCC_DUPLICATE).nonzero().cpu().numpy()
    ncc_links += [(start + a, b) for a, b in hit if start + a < b]
kept_dist = lib.hamming_matrix(HASHES[kept_index], HASHES[kept_index])
ham_links = list(zip(*np.nonzero(np.triu(kept_dist <= GROUP_TOLERANCE, 1))))
groups = lib.connected_groups(ham_links + ncc_links, len(kept_index))
group_sizes = pd.Series(groups).value_counts()
log(
    f"{len(kept_index)} images in {len(group_sizes)} groups; largest group {group_sizes.iloc[0]} images; "
    f"{int((group_sizes > 1).sum())} groups with more than one image"
)
if group_sizes.iloc[0] > 0.05 * len(kept_index):
    log("Warning: one group holds more than 5% of the pool; consider a smaller GROUP_TOLERANCE.")

labels_kept = frame["label"].to_numpy()[kept_index]
outer = StratifiedGroupKFold(n_splits=round(1 / TEST_FRACTION), shuffle=True, random_state=SEED)
dev_pos, test_pos = next(outer.split(np.zeros(len(kept_index)), labels_kept, groups))
frame.loc[kept_index[test_pos], "role"] = "test"
frame.loc[kept_index[dev_pos], "role"] = "dev"
frame["fold"] = -1
inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, val_pos) in enumerate(
    inner.split(np.zeros(len(dev_pos)), labels_kept[dev_pos], groups[dev_pos])
):
    frame.loc[kept_index[dev_pos[val_pos]], "fold"] = fold
frame["group"] = -1
frame.loc[kept_index, "group"] = groups

# Leakage audit: nearest development image for every test image
dev_index, test_index = kept_index[dev_pos], kept_index[test_pos]
test_dev_dist = lib.hamming_matrix(HASHES[test_index], HASHES[dev_index])
test_dev_ncc = (
    (
        torch.from_numpy(NCC_VEC[test_index]).to(DEVICE)
        @ torch.from_numpy(NCC_VEC[dev_index]).to(DEVICE).T
    )
    .max(dim=1)
    .values.cpu()
    .numpy()
)
frame["nearest_dev_hamming"] = np.nan
frame["max_dev_ncc"] = np.nan
frame.loc[test_index, "nearest_dev_hamming"] = test_dev_dist.min(axis=1)
frame.loc[test_index, "max_dev_ncc"] = test_dev_ncc
audit = pd.DataFrame(
    {
        "criterion": [
            f"Hamming <= {DEDUP_TOLERANCE}",
            f"Hamming <= {GROUP_TOLERANCE}",
            f"NCC >= {NCC_DUPLICATE}",
        ],
        "test images with a development neighbour": [
            int((test_dev_dist.min(axis=1) <= DEDUP_TOLERANCE).sum()),
            int((test_dev_dist.min(axis=1) <= GROUP_TOLERANCE).sum()),
            int((test_dev_ncc >= NCC_DUPLICATE).sum()),
        ],
    }
)
save_table(audit, "leakage_audit")
assert (
    audit["test images with a development neighbour"].sum() == 0
), "Leakage between test and development images"

# Optional external images: drop any near an image of the pool
external = np.flatnonzero(((frame["origin"] == "external") & frame["ok"]).to_numpy())
if len(external):
    ext_dist = lib.hamming_matrix(HASHES[external], HASHES[POOL]).min(axis=1)
    ext_ncc = (
        (
            torch.from_numpy(NCC_VEC[external]).to(DEVICE)
            @ torch.from_numpy(NCC_VEC[POOL]).to(DEVICE).T
        )
        .max(dim=1)
        .values.cpu()
        .numpy()
    )
    near = (ext_dist <= GROUP_TOLERANCE) | (ext_ncc >= NCC_DUPLICATE)
    frame.loc[external[near], "role"] = "external duplicate"
    log(
        f"External images: {len(external)}, {int(near.sum())} removed as near-duplicates of the pool, {int((~near).sum())} kept"
    )
del vectors
torch.cuda.empty_cache()

split_table = pd.crosstab(frame["role"], frame["class_name"], margins=True, margins_name="Total")
fold_table = pd.crosstab(
    frame.loc[frame["role"] == "dev", "fold"],
    frame.loc[frame["role"] == "dev", "class_name"],
    margins=True,
)
save_table(split_table, "split_roles", index=True)
save_table(fold_table, "fold_distribution", index=True)
frame.to_csv(METRICS / "splits.csv", index=False)
SPLIT_TABLE = METRICS / "splits.csv"

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
group_sizes.value_counts().sort_index().plot.bar(ax=axes[0], logy=True)
axes[0].set(title="Group sizes", xlabel="Images in group", ylabel="Groups (log)")
axes[1].hist(frame.loc[test_index, "nearest_dev_hamming"], bins=np.arange(0, 40), color="tab:blue")
axes[1].axvline(GROUP_TOLERANCE + 0.5, color="red", linestyle="--", label="grouping tolerance")
axes[1].set(
    title="Nearest development image of each test image",
    xlabel="Hamming distance",
    ylabel="Test images",
)
axes[1].legend()
fig.tight_layout()
save_fig(fig, "05_groups_and_leakage")

### Augmentation preview

The GPU augmentation applied during training (flip, rotation up to 10 degrees, brightness and contrast jitter of 20 percent), shown on the CPU for a few training images, together with the two masked inputs of the shortcut tests. Expected output: one figure.

In [ ]:
preview = frame[frame["role"] == "dev"].groupby("label").head(2)
batch = torch.from_numpy(np.ascontiguousarray(IMAGES[preview["cache_index"].to_numpy()]))
batch_masks = torch.from_numpy(np.ascontiguousarray(MASKS[preview["cache_index"].to_numpy()]))
cpu_prep = lib.GpuPreprocessor(224, torch.device("cpu"))
torch.manual_seed(SEED)
variants = {
    "Evaluation input": cpu_prep(batch),
    "Augmented": cpu_prep(batch, train=True),
    "Augmented again": cpu_prep(batch, train=True),
    "Background only": cpu_prep(batch, batch_masks, mode="background"),
    "Head only": cpu_prep(batch, batch_masks, mode="head"),
}
fig, axes = plt.subplots(
    len(variants), len(preview), figsize=(2.6 * len(preview), 2.6 * len(variants)), squeeze=False
)
for row, (title, tensors) in enumerate(variants.items()):
    for col in range(len(preview)):
        axes[row, col].imshow(lib.denormalise(tensors[col]))
        axes[row, col].set_title(
            (
                f"{title}\n{CLASS_NAMES[preview['label'].iloc[col]]}"
                if col == 0
                else CLASS_NAMES[preview["label"].iloc[col]]
            ),
            fontsize=8,
        )
        axes[row, col].axis("off")
fig.tight_layout()
save_fig(fig, "06_augmentation")

## 5. Training

### Download pretrained weights and profile every model

Downloads each ImageNet checkpoint once in this process (so the two workers never race on the same download) and records parameter count and GFLOPs, at the native input and at 224 px so InceptionV3 can also be compared like for like. Jobs are then ordered most expensive first, which balances the two GPUs. Expected output: one row per model.

In [ ]:
profiles = []
for name in tqdm(MODELS, desc="Pretrained weights"):
    model = lib.build_model(name, NUM_OUTPUTS, pretrained=True)
    spec = lib.MODEL_SPECS[name]
    profiles.append(
        {
            "model": name,
            "family": spec["family"],
            "input_size": spec["input_size"],
            "parameters_m": lib.count_parameters(model) / 1e6,
            "gflops_native": lib.gflops(model, spec["input_size"]),
            "gflops_224": lib.gflops(model, 224),
        }
    )
    del model
PROFILES = pd.DataFrame(profiles).set_index("model")
save_table(PROFILES.round(3), "model_profiles", index=True)

### Launch the two GPU workers

Builds the job list (every model x fold, plus one background-only run per model when `SHORTCUT_TEST` is on), restores finished runs from `RESUME_DIR` if given (after checking the splits are identical), writes the worker configuration, and starts one worker per GPU. The cell then follows both workers live: an overall jobs bar, a step bar per GPU with loss, learning rate and peak memory, and one line per finished epoch. Worker logs are in `logs/`. Expected output: live progress until all jobs finish or the time budget is reached.

In [ ]:
JOBS = [{"model": m, "fold": f, "mode": "full"} for m in MODELS for f in range(N_FOLDS)]
if SHORTCUT_TEST:
    JOBS += [{"model": m, "fold": 0, "mode": "background"} for m in MODELS]
cost = PROFILES["gflops_native"].to_dict()
JOBS.sort(key=lambda job: -cost[job["model"]])

if RESUME_DIR is not None:
    previous = pd.read_csv(Path(RESUME_DIR) / "metrics" / "splits.csv")
    current = pd.read_csv(SPLIT_TABLE)
    same = previous[["path", "role", "fold"]].equals(current[["path", "role", "fold"]])
    assert same, "RESUME_DIR was produced with different splits; it cannot be resumed."
    for sub in ("metrics/runs", "predictions", "weights"):
        if (Path(RESUME_DIR) / sub).exists():
            shutil.copytree(Path(RESUME_DIR) / sub, WORK / sub, dirs_exist_ok=True)
    log(f"Restored {len(list(RUNS.glob('*/summary.json')))} finished runs from {RESUME_DIR}")

shutil.rmtree(CLAIMS, ignore_errors=True)
CLAIMS.mkdir(parents=True)
for path in LOGS.glob("status_gpu*.json"):
    path.unlink()
worker_config = {
    "num_outputs": NUM_OUTPUTS,
    "seed": SEED,
    "deterministic": DETERMINISTIC,
    "effective_batch": EFFECTIVE_BATCH,
    "epochs": EPOCHS,
    "warmup_epochs": WARMUP_EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "grad_clip": GRAD_CLIP,
    "runs_dir": str(RUNS),
    "weights_dir": str(WEIGHTS),
    "predictions_dir": str(PREDICTIONS),
    "log_dir": str(LOGS),
    "claims_dir": str(CLAIMS),
    "split_table": str(SPLIT_TABLE),
    "images_path": str(IMAGES_PATH),
    "masks_path": str(MASKS_PATH),
    "deadline": time.time() + TRAINING_BUDGET_HOURS * 3600,
    "jobs": JOBS,
}
CONFIG_PATH = METRICS / "worker_config.json"
shared_lib.write_json(CONFIG_PATH, worker_config)


def finished_jobs():
    return sum(
        (RUNS / lib.run_id(j["model"], j["fold"], j["mode"]) / "summary.json").exists()
        for j in JOBS
    )


log(f"{len(JOBS)} jobs, {finished_jobs()} already finished; launching {GPU_COUNT} workers")
training_start = time.perf_counter()
workers = shared_lib.launch_workers(SRC / "image_worker.py", CONFIG_PATH, LOGS, GPU_COUNT)
shared_lib.monitor_workers(workers, LOGS, len(JOBS), finished_jobs)
TRAINING_WALL_SECONDS = time.perf_counter() - training_start
log(
    f"Training stage finished in {TRAINING_WALL_SECONDS / 60:.1f} min; {finished_jobs()}/{len(JOBS)} jobs complete"
)

## 6. Results

### Per-run calibration and metrics

For every finished run: fit a temperature on its own validation fold, choose the binary decision threshold on that fold (Youden's J), then score the test set. Folds are summarised as mean ± standard deviation. A model missing folds (time budget) is flagged rather than dropped. Expected output: the per-model fold summary.

In [ ]:
SUMMARIES = pd.DataFrame([shared_lib.read_json(p) for p in sorted(RUNS.glob("*/summary.json"))])
assert len(SUMMARIES), "No finished runs: training did not complete any job."
LABEL_OF = frame["label"].to_numpy()
BINARY = NUM_CLASSES == 2
run_rows, RUN_PROBS = [], {}
for summary in tqdm(SUMMARIES.to_dict("records"), desc="Scoring runs"):
    data = np.load(PREDICTIONS / f"{summary['run_id']}.npz")
    y_val, y_test = LABEL_OF[data["val_index"]], LABEL_OF[data["test_index"]]
    temperature = lib.fit_temperature(data["val_logits"], y_val)
    val_probs = lib.logits_to_probs(data["val_logits"], temperature)
    test_probs = lib.logits_to_probs(data["test_logits"], temperature)
    threshold = lib.youden_threshold(y_val, val_probs[:, 1]) if BINARY else None
    RUN_PROBS[summary["run_id"]] = {
        "test_index": data["test_index"],
        "probs": test_probs,
        "uncalibrated": lib.logits_to_probs(data["test_logits"]),
        "threshold": threshold,
        **{key: data[key] for key in data.files if key.startswith(("test_logits_", "external_"))},
    }
    row = {**summary, "temperature": temperature, "threshold": threshold}
    row.update({f"test_{k}": v for k, v in lib.classification_metrics(y_test, test_probs).items()})
    row["test_ece_uncalibrated"] = lib.expected_calibration_error(
        y_test, lib.logits_to_probs(data["test_logits"])
    )
    if BINARY:
        tuned = lib.classification_metrics(y_test, test_probs, threshold)
        row.update(
            {
                f"test_tuned_{k}": tuned[k]
                for k in ("accuracy", "sensitivity", "specificity", "f1_macro")
            }
        )
    run_rows.append(row)
RUN_METRICS = pd.DataFrame(run_rows)
save_table(RUN_METRICS, "run_metrics", show=False)

FULL = RUN_METRICS[RUN_METRICS["mode"] == "full"]
fold_cols = [
    "test_accuracy",
    "test_balanced_accuracy",
    "test_f1_macro",
    "test_roc_auc",
    "test_ece",
    "test_ece_uncalibrated",
    "test_log_loss",
]
if BINARY:
    fold_cols += [
        "test_sensitivity",
        "test_specificity",
        "test_tuned_accuracy",
        "test_tuned_sensitivity",
        "test_tuned_specificity",
    ]
fold_summary = FULL.groupby("model")[fold_cols].agg(["mean", "std"])
fold_summary.columns = [f"{a}_{b}" for a, b in fold_summary.columns]
fold_summary.insert(0, "folds", FULL.groupby("model").size())
incomplete = fold_summary.index[fold_summary["folds"] < N_FOLDS].tolist()
if incomplete:
    log(f"Models with missing folds (rerun with RESUME_DIR): {incomplete}")
save_table(
    fold_summary.sort_values("test_accuracy_mean", ascending=False).round(4),
    "fold_summary",
    index=True,
)

### Training curves

Mean ± one standard deviation across folds of training and validation loss, validation accuracy and the learning rate, per model. Expected output: one figure with a row per model.

In [ ]:
histories = {
    rid: pd.read_csv(RUNS / rid / "history.csv")
    for rid in SUMMARIES.loc[SUMMARIES["mode"] == "full", "run_id"]
}
models_done = [m for m in MODELS if m in FULL["model"].unique()]
fig, axes = plt.subplots(len(models_done), 3, figsize=(17, 2.8 * len(models_done)), squeeze=False)
for row, name in enumerate(models_done):
    runs = [histories[rid] for rid in FULL.loc[FULL["model"] == name, "run_id"]]
    stacked = pd.concat(runs).groupby("epoch")
    mean, std = stacked.mean(), stacked.std().fillna(0)
    for metric, colour in (("train_loss", "tab:blue"), ("val_loss", "tab:orange")):
        axes[row, 0].plot(mean.index, mean[metric], color=colour, label=metric)
        axes[row, 0].fill_between(
            mean.index,
            mean[metric] - std[metric],
            mean[metric] + std[metric],
            color=colour,
            alpha=0.2,
        )
    axes[row, 1].plot(mean.index, mean["train_acc"], label="train_acc")
    axes[row, 1].plot(mean.index, mean["val_acc"], label="val_acc")
    axes[row, 1].fill_between(
        mean.index,
        mean["val_acc"] - std["val_acc"],
        mean["val_acc"] + std["val_acc"],
        alpha=0.2,
        color="tab:orange",
    )
    axes[row, 2].plot(runs[0]["epoch"], runs[0]["lr"], color="tab:purple")
    axes[row, 0].set_ylabel(name)
    axes[row, 0].legend(fontsize=7)
    axes[row, 1].legend(fontsize=7)
axes[0, 0].set_title("Loss (mean ± sd over folds)")
axes[0, 1].set_title("Accuracy")
axes[0, 2].set_title("Learning rate at epoch end")
fig.tight_layout()
save_fig(fig, "07_training_curves")

### Fold-ensemble test results with bootstrap confidence intervals

Each model's calibrated test probabilities are averaged over its folds, giving one prediction per test image per model. Accuracy, macro F1 and AUC get percentile bootstrap 95% CIs over test images. For binary detection the table also reports sensitivity and specificity at 0.5 and at the mean validation-tuned threshold. Expected output: the leaderboard and the per-class table.

In [ ]:
TEST_INDEX = RUN_PROBS[FULL["run_id"].iloc[0]]["test_index"]
Y_TEST = LABEL_OF[TEST_INDEX]


def auc_score(y, p):
    return (
        skm.roc_auc_score(y, p[:, 1])
        if BINARY
        else skm.roc_auc_score(y, p, multi_class="ovr", average="macro")
    )


# Fast metric functions for the bootstrap (thousands of resamples per model)
BOOT_METRICS = {
    "accuracy": lambda y, p, t: float((p.argmax(1) == y).mean()),
    "f1_macro": lambda y, p, t: skm.f1_score(
        y, p.argmax(1), labels=range(NUM_CLASSES), average="macro", zero_division=0
    ),
    "roc_auc": lambda y, p, t: auc_score(y, p),
}
ENSEMBLE, leaderboard, per_class = {}, [], []
for name in models_done:
    ids = FULL.loc[FULL["model"] == name, "run_id"]
    assert all(np.array_equal(RUN_PROBS[r]["test_index"], TEST_INDEX) for r in ids)
    probs = np.mean([RUN_PROBS[r]["probs"] for r in ids], axis=0)
    threshold = float(np.mean([RUN_PROBS[r]["threshold"] for r in ids])) if BINARY else None
    ENSEMBLE[name] = {"probs": probs, "threshold": threshold}
    metrics = lib.classification_metrics(Y_TEST, probs)
    row = {"model": name, "family": lib.MODEL_SPECS[name]["family"], "folds": len(ids), **metrics}
    for key, fn in BOOT_METRICS.items():
        low, high = lib.bootstrap_ci(Y_TEST, probs, fn, BOOTSTRAP_SAMPLES, SEED)
        row[f"{key}_ci"] = f"[{low:.4f}, {high:.4f}]"
    if BINARY:
        tuned = lib.classification_metrics(Y_TEST, probs, threshold)
        row.update(
            {
                "threshold": threshold,
                "tuned_accuracy": tuned["accuracy"],
                "tuned_sensitivity": tuned["sensitivity"],
                "tuned_specificity": tuned["specificity"],
            }
        )
    leaderboard.append(row)
    per_class.append(
        lib.per_class_metrics(Y_TEST, lib.predictions_from_probs(probs), probs, CLASS_NAMES).assign(
            model=name
        )
    )
LEADERBOARD = (
    pd.DataFrame(leaderboard)
    .sort_values(["accuracy", "roc_auc"], ascending=False)
    .set_index("model")
)
LEADERBOARD = LEADERBOARD.join(PROFILES[["parameters_m", "gflops_native"]])
save_table(LEADERBOARD.round(4), "test_leaderboard", index=True)
save_table(pd.concat(per_class).round(4), "test_per_class")

### Statistical comparison

Exact McNemar tests on every pair of models' fold-ensemble test predictions, Holm-corrected, and for each model the bootstrap 95% CI of its accuracy difference from the top model. Models whose interval includes zero cannot be separated from the leader on this test set. Expected output: a table and a p-value heatmap.

In [ ]:
correct = {
    name: lib.predictions_from_probs(ENSEMBLE[name]["probs"]) == Y_TEST for name in models_done
}
pairs = [(a, b) for i, a in enumerate(models_done) for b in models_done[i + 1 :]]
raw_p = [lib.mcnemar_p(correct[a], correct[b]) for a, b in pairs]
adjusted = shared_lib.holm(raw_p)
P_MATRIX = pd.DataFrame(1.0, index=models_done, columns=models_done)
for (a, b), p in zip(pairs, adjusted):
    P_MATRIX.loc[a, b] = P_MATRIX.loc[b, a] = p
save_table(P_MATRIX.round(4), "mcnemar_holm_p", index=True, show=False)

leader = LEADERBOARD.index[0]
rng = np.random.default_rng(SEED)
boots = rng.integers(0, len(Y_TEST), (BOOTSTRAP_SAMPLES, len(Y_TEST)))
diff_rows = []
for name in models_done:
    diffs = correct[leader][boots].mean(axis=1) - correct[name][boots].mean(axis=1)
    low, high = np.percentile(diffs, [2.5, 97.5])
    diff_rows.append(
        {
            "model": name,
            "accuracy_gap_to_leader": correct[leader].mean() - correct[name].mean(),
            "ci_low": low,
            "ci_high": high,
            "separable_from_leader": low > 0,
            "mcnemar_holm_p": P_MATRIX.loc[leader, name],
        }
    )
save_table(pd.DataFrame(diff_rows).round(4), "gap_to_leader")

fig, ax = plt.subplots(figsize=(1.0 * len(models_done) + 3, 0.8 * len(models_done) + 2))
image = ax.imshow(-np.log10(P_MATRIX.to_numpy().clip(1e-12)), cmap="viridis")
ax.set_xticks(range(len(models_done)), models_done, rotation=45, ha="right")
ax.set_yticks(range(len(models_done)), models_done)
for i in range(len(models_done)):
    for j in range(len(models_done)):
        ax.text(
            j, i, f"{P_MATRIX.iat[i, j]:.3f}", ha="center", va="center", fontsize=7, color="white"
        )
ax.grid(False)
plt.colorbar(image, ax=ax, label="-log10 Holm-adjusted p")
ax.set_title("Pairwise McNemar tests on test predictions")
fig.tight_layout()
save_fig(fig, "08_mcnemar")

### Confusion matrices

Test-set confusion matrices of every model's fold ensemble, counts with row percentages. Expected output: one grid figure.

In [ ]:
cols = 3
rows_n = math.ceil(len(models_done) / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(5.2 * cols, 4.4 * rows_n), squeeze=False)
for ax, name in zip(axes.flat, models_done):
    cm = skm.confusion_matrix(
        Y_TEST, lib.predictions_from_probs(ENSEMBLE[name]["probs"]), labels=range(NUM_CLASSES)
    )
    share = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(
                j,
                i,
                f"{cm[i, j]}\n{share[i, j]:.1%}",
                ha="center",
                va="center",
                color="white" if share[i, j] > 0.5 else "black",
                fontsize=9,
            )
    ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=20)
    ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
    ax.set(
        title=f"{name} (acc {LEADERBOARD.loc[name, 'accuracy']:.4f})",
        xlabel="Predicted",
        ylabel="True",
    )
    ax.grid(False)
for ax in axes.flat[len(models_done) :]:
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "09_confusion_matrices")

### ROC and precision-recall curves

One-versus-rest curves of every model's fold ensemble on the test set (the tumour class for binary detection; the macro view for subtypes). Expected output: a two-panel figure per class.

In [ ]:
curve_classes = [1] if BINARY else list(range(NUM_CLASSES))
fig, axes = plt.subplots(
    len(curve_classes), 2, figsize=(14, 5.5 * len(curve_classes)), squeeze=False
)
for row, k in enumerate(curve_classes):
    positive = (Y_TEST == k).astype(int)
    for name in models_done:
        p = ENSEMBLE[name]["probs"][:, k]
        fpr, tpr, _ = skm.roc_curve(positive, p)
        precision, recall, _ = skm.precision_recall_curve(positive, p)
        axes[row, 0].plot(fpr, tpr, label=f"{name} ({skm.auc(fpr, tpr):.4f})")
        axes[row, 1].plot(
            recall, precision, label=f"{name} ({skm.average_precision_score(positive, p):.4f})"
        )
    axes[row, 0].plot([0, 1], [0, 1], "k--", linewidth=0.8)
    axes[row, 0].set(
        title=f"ROC: {CLASS_NAMES[k]} vs rest",
        xlabel="False positive rate",
        ylabel="True positive rate",
    )
    axes[row, 1].set(
        title=f"Precision-recall: {CLASS_NAMES[k]} vs rest", xlabel="Recall", ylabel="Precision"
    )
    axes[row, 0].legend(fontsize=7, loc="lower right")
    axes[row, 1].legend(fontsize=7, loc="lower left")
fig.tight_layout()
save_fig(fig, "10_roc_pr_curves")

### Calibration before and after temperature scaling

Reliability diagrams of each model's fold ensemble on the test set, uncalibrated and temperature-scaled (temperatures fitted on validation folds only), with the expected calibration error of each. Expected output: one grid figure and an ECE table.

In [ ]:
fig, axes = plt.subplots(rows_n, cols, figsize=(5.2 * cols, 4.4 * rows_n), squeeze=False)
ece_rows = []
for ax, name in zip(axes.flat, models_done):
    ids = FULL.loc[FULL["model"] == name, "run_id"]
    raw = np.mean([RUN_PROBS[r]["uncalibrated"] for r in ids], axis=0)
    for probs, label, colour in (
        (raw, "uncalibrated", "tab:red"),
        (ENSEMBLE[name]["probs"], "temperature-scaled", "tab:green"),
    ):
        table = lib.calibration_table(Y_TEST, probs).dropna()
        ece = lib.expected_calibration_error(Y_TEST, probs)
        ax.plot(
            table["mean_confidence"],
            table["accuracy"],
            marker="o",
            color=colour,
            label=f"{label} (ECE {ece:.4f})",
        )
        ece_rows.append({"model": name, "variant": label, "ece": ece})
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
    ax.set(
        title=name,
        xlabel="Confidence",
        ylabel="Accuracy",
        xlim=(0.4 if BINARY else 0.3, 1.01),
        ylim=(0.3, 1.01),
    )
    ax.legend(fontsize=7, loc="upper left")
for ax in axes.flat[len(models_done) :]:
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "11_calibration")
save_table(
    pd.DataFrame(ece_rows).pivot(index="model", columns="variant", values="ece").round(4),
    "calibration_ece",
    index=True,
)

### Accuracy against distance to the development pool

Even after group-aware splitting, some test images are closer to development images than others. Test accuracy is reported by the Hamming distance of each test image's nearest development image, so any residual dependence on near-copies is visible. Expected output: a table and a figure.

In [ ]:
distance = frame.loc[TEST_INDEX, "nearest_dev_hamming"].to_numpy()
bins = [GROUP_TOLERANCE, 12, 16, 20, 64]
labels_b = [f"{a + 1}-{b}" for a, b in zip(bins[:-1], bins[1:])]
bin_of = pd.cut(distance, bins=bins, labels=labels_b)
leak_rows = []
for name in models_done:
    for label_b in labels_b:
        mask = np.asarray(bin_of == label_b)
        if mask.any():
            leak_rows.append(
                {
                    "model": name,
                    "distance_bin": label_b,
                    "images": int(mask.sum()),
                    "accuracy": float(correct[name][mask].mean()),
                }
            )
LEAK = pd.DataFrame(leak_rows)
save_table(
    LEAK.pivot(index="model", columns="distance_bin", values="accuracy").round(4),
    "accuracy_by_nearest_dev_distance",
    index=True,
)
fig, ax = plt.subplots(figsize=(11, 4.5))
for name in models_done:
    sub = LEAK[LEAK["model"] == name]
    ax.plot(sub["distance_bin"].astype(str), sub["accuracy"], marker="o", label=name)
ax.set(
    title="Test accuracy by Hamming distance to the nearest development image",
    xlabel="Distance bin (bits)",
    ylabel="Accuracy",
)
ax.legend(fontsize=7, ncol=3)
save_fig(fig, "12_accuracy_by_distance")

## 7. Interpretability

### Grad-CAM, attention rollout and outside-head energy for every model

For each model, the fold with the lowest validation loss is reloaded and Grad-CAM is computed for **every** test image (batched; samples are independent in evaluation mode). The share of map energy outside the head mask is reported per class as mean ± sd, next to the share a uniform map would give. ViT-B/16 also gets attention rollout as an attention-native check. Expected output: the outside-energy table, a bar chart and one gallery per model of its most confident correct and most confident wrong predictions.

In [ ]:
def best_fold_run(name):
    sub = FULL[FULL["model"] == name]
    return sub.loc[sub["best_val_loss"].idxmin()]


def load_trained(name):
    run = best_fold_run(name)
    model = lib.build_model(name, NUM_OUTPUTS, pretrained=False)
    lib.load_weights(model, WEIGHTS / lib.slug(name) / f"{run['run_id']}.pth", "cpu")
    return model.to(DEVICE).eval(), run


test_images = IMAGES[TEST_INDEX]
test_masks = MASKS[TEST_INDEX]
outside_rows, GRADCAM_STORE = [], {}
for name in tqdm(models_done, desc="Grad-CAM models"):
    model, run = load_trained(name)
    size = lib.MODEL_SPECS[name]["input_size"]
    prep = lib.GpuPreprocessor(size, DEVICE)
    layer = lib.gradcam_layer(model, name)
    fractions, rollout_fractions, maps_keep = [], [], {}
    for start in tqdm(range(0, len(TEST_INDEX), GRADCAM_BATCH), desc=name, leave=False):
        batch = torch.from_numpy(np.ascontiguousarray(test_images[start : start + GRADCAM_BATCH]))
        masks = torch.from_numpy(
            np.ascontiguousarray(test_masks[start : start + GRADCAM_BATCH])
        ).to(DEVICE)
        masks = F.interpolate(masks[:, None].float(), size=(size, size), mode="nearest")[
            :, 0
        ].bool()
        inputs = prep(batch)
        maps, _ = lib.gradcam_batch(model, layer, inputs, BINARY)
        fractions.append(lib.outside_fraction(maps, masks).cpu())
        if name == "ViT-B/16":
            rollout_fractions.append(
                lib.outside_fraction(lib.vit_rollout(model, inputs), masks).cpu()
            )
    fractions = torch.cat(fractions).numpy()
    uniform = 1 - test_masks.reshape(len(test_masks), -1).mean(
        axis=1
    )  # outside share of a flat map
    for label, cls in enumerate(CLASS_NAMES):
        sel = Y_TEST == label
        outside_rows.append(
            {
                "model": name,
                "class": cls,
                "method": "Grad-CAM",
                "mean": np.nanmean(fractions[sel]),
                "sd": np.nanstd(fractions[sel]),
                "uniform_map": uniform[sel].mean(),
            }
        )
        if rollout_fractions:
            rf = torch.cat(rollout_fractions).numpy()
            outside_rows.append(
                {
                    "model": name,
                    "class": cls,
                    "method": "Attention rollout",
                    "mean": np.nanmean(rf[sel]),
                    "sd": np.nanstd(rf[sel]),
                    "uniform_map": uniform[sel].mean(),
                }
            )
    # Gallery: most confident correct (2 per class) and most confident errors (up to 6)
    probs = RUN_PROBS[run["run_id"]]["probs"]
    predicted = lib.predictions_from_probs(probs)
    confidence = probs.max(axis=1)
    picks = []
    for label in range(NUM_CLASSES):
        ok = np.flatnonzero((predicted == Y_TEST) & (Y_TEST == label))
        picks += list(ok[np.argsort(-confidence[ok])][:2])
    wrong = np.flatnonzero(predicted != Y_TEST)
    picks += list(wrong[np.argsort(-confidence[wrong])][:6])
    inputs = prep(torch.from_numpy(np.ascontiguousarray(test_images[picks])))
    maps, _ = lib.gradcam_batch(model, layer, inputs, BINARY)
    fig, axes = plt.subplots(2, len(picks), figsize=(2.3 * len(picks), 5), squeeze=False)
    for col, index in enumerate(picks):
        image = lib.denormalise(inputs[col])
        axes[0, col].imshow(image)
        axes[1, col].imshow(image)
        axes[1, col].imshow(maps[col].cpu().numpy(), cmap="jet", alpha=0.45, vmin=0, vmax=1)
        colour = "green" if predicted[index] == Y_TEST[index] else "red"
        axes[0, col].set_title(
            f"T: {CLASS_NAMES[Y_TEST[index]]}\nP: {CLASS_NAMES[predicted[index]]} ({confidence[index]:.2f})",
            fontsize=7,
            color=colour,
        )
        for ax in axes[:, col]:
            ax.axis("off")
    fig.suptitle(
        f"{name} Grad-CAM: most confident correct (left) and most confident errors (right)"
    )
    fig.tight_layout()
    save_fig(fig, f"13_gradcam_{lib.slug(name)}")
    del model
    torch.cuda.empty_cache()

OUTSIDE = pd.DataFrame(outside_rows)
save_table(OUTSIDE.round(4), "outside_head_energy")
fig, ax = plt.subplots(figsize=(13, 4.5))
grad = OUTSIDE[OUTSIDE["method"] == "Grad-CAM"]
width = 0.8 / NUM_CLASSES
for k, cls in enumerate(CLASS_NAMES):
    sub = grad[grad["class"] == cls].set_index("model").reindex(models_done)
    x = np.arange(len(models_done)) + (k - (NUM_CLASSES - 1) / 2) * width
    ax.bar(x, sub["mean"], width, yerr=sub["sd"], capsize=2, color=CLASS_COLOURS[k], label=cls)
    ax.scatter(x, sub["uniform_map"], marker="_", s=200, color="black")
ax.set_xticks(range(len(models_done)), models_done, rotation=30, ha="right")
ax.set(
    title="Share of Grad-CAM energy outside the head (black mark: uniform map)",
    ylabel="Outside-head share",
)
ax.legend()
save_fig(fig, "14_outside_head_energy")

### Occlusion maps (model-agnostic)

Grad-CAM maps of convolutional and attention stages are not strictly commensurable. Occlusion sensitivity (the drop in the predicted-class probability when a patch is replaced by the image mean) is computed the same way for every architecture, on the same stratified sample of test images. Expected output: outside-head shares of the occlusion maps and a models x images gallery.

In [ ]:
rng = np.random.default_rng(SEED)
occ_pick = np.concatenate(
    [
        rng.choice(
            np.flatnonzero(Y_TEST == k),
            min(OCCLUSION_IMAGES_PER_CLASS, int((Y_TEST == k).sum())),
            replace=False,
        )
        for k in range(NUM_CLASSES)
    ]
)
show = np.concatenate([occ_pick[Y_TEST[occ_pick] == k][:2] for k in range(NUM_CLASSES)])
occ_rows = []
fig, axes = plt.subplots(
    len(models_done), len(show), figsize=(2.1 * len(show), 2.1 * len(models_done)), squeeze=False
)
for row, name in enumerate(tqdm(models_done, desc="Occlusion")):
    model, _ = load_trained(name)
    size = lib.MODEL_SPECS[name]["input_size"]
    prep = lib.GpuPreprocessor(size, DEVICE)
    inputs = prep(torch.from_numpy(np.ascontiguousarray(test_images[occ_pick])))
    masks = F.interpolate(
        torch.from_numpy(np.ascontiguousarray(test_masks[occ_pick])).to(DEVICE)[:, None].float(),
        size=(size, size),
        mode="nearest",
    )[:, 0].bool()
    maps, _ = lib.occlusion_maps(model, inputs, BINARY, OCCLUSION_PATCH, OCCLUSION_STRIDE)
    frac = lib.outside_fraction(maps, masks).cpu().numpy()
    occ_rows.append(
        {
            "model": name,
            "outside_mean": np.nanmean(frac),
            "outside_sd": np.nanstd(frac),
            "images": len(occ_pick),
        }
    )
    for col, index in enumerate(show):
        position = int(np.flatnonzero(occ_pick == index)[0])
        axes[row, col].imshow(lib.denormalise(inputs[position]))
        axes[row, col].imshow(maps[position].cpu().numpy(), cmap="jet", alpha=0.45, vmin=0, vmax=1)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title(CLASS_NAMES[Y_TEST[index]], fontsize=8)
    axes[row, 0].set_ylabel(name, fontsize=8)
    del model
    torch.cuda.empty_cache()
fig.suptitle("Occlusion sensitivity on the same test images for every model")
fig.tight_layout()
save_fig(fig, "15_occlusion_maps")
save_table(pd.DataFrame(occ_rows).round(4), "occlusion_outside_head")

### Masking test (inference only)

Each fold's checkpoint is applied to test images with the head set to zero (background only) and with the background set to zero (head only). Accuracy near the majority-class rate on background-only inputs means the model needs the head; a large drop on head-only inputs would mean it relies on the background. Expected output: a table and a bar chart.

In [ ]:
majority = np.bincount(Y_TEST, minlength=NUM_CLASSES).max() / len(Y_TEST)
mask_rows = []
for run in FULL.to_dict("records"):
    store = RUN_PROBS[run["run_id"]]
    for key, variant in (
        ("test_logits_background_only", "background only"),
        ("test_logits_head_only", "head only"),
    ):
        probs = lib.logits_to_probs(store[key], run["temperature"])
        mask_rows.append(
            {
                "model": run["model"],
                "fold": run["fold"],
                "input": variant,
                "accuracy": float((lib.predictions_from_probs(probs) == Y_TEST).mean()),
            }
        )
    mask_rows.append(
        {
            "model": run["model"],
            "fold": run["fold"],
            "input": "full image",
            "accuracy": run["test_accuracy"],
        }
    )
MASKING = (
    pd.DataFrame(mask_rows).groupby(["model", "input"])["accuracy"].agg(["mean", "std"]).unstack()
)
save_table(MASKING.round(4), "masking_test", index=True)
fig, ax = plt.subplots(figsize=(13, 4.5))
variants_m = ["full image", "head only", "background only"]
for k, variant in enumerate(variants_m):
    values = MASKING[("mean", variant)].reindex(models_done)
    errors = MASKING[("std", variant)].reindex(models_done)
    ax.bar(
        np.arange(len(models_done)) + (k - 1) * 0.27,
        values,
        0.27,
        yerr=errors,
        capsize=2,
        label=variant,
    )
ax.axhline(majority, color="black", linestyle="--", label=f"majority-class rate ({majority:.3f})")
ax.set_xticks(range(len(models_done)), models_done, rotation=30, ha="right")
ax.set(
    title="Test accuracy with masked inputs (mean ± sd over folds)",
    ylabel="Accuracy",
    ylim=(0, 1.02),
)
ax.legend(fontsize=8)
save_fig(fig, "16_masking_test")

### Shortcut test: retraining on background-only images

The decisive check the reviewers asked for. Each model was also trained (fold 0) on images whose head region is set to zero, leaving only background, border and framing, and tested on background-only test images. Accuracy significantly above the majority-class rate (one-sided exact binomial test, Holm-corrected) means the background carries label information a model can exploit. Expected output: a table and a bar chart, or a note if `SHORTCUT_TEST` is off.

In [ ]:
BACKGROUND = RUN_METRICS[RUN_METRICS["mode"] == "background"]
if BACKGROUND.empty:
    log("No background-only runs (SHORTCUT_TEST is off or the runs did not finish).")
else:
    rows_s = []
    for run in BACKGROUND.to_dict("records"):
        hits = int(round(run["test_accuracy"] * len(Y_TEST)))
        rows_s.append(
            {
                "model": run["model"],
                "background_only_accuracy": run["test_accuracy"],
                "background_only_auc": run["test_roc_auc"],
                "majority_rate": majority,
                "p_value": binomtest(hits, len(Y_TEST), majority, alternative="greater").pvalue,
            }
        )
    SHORTCUT = (
        pd.DataFrame(rows_s)
        .set_index("model")
        .reindex([m for m in models_done if m in set(BACKGROUND["model"])])
    )
    SHORTCUT["p_holm"] = shared_lib.holm(SHORTCUT["p_value"].to_numpy())
    SHORTCUT["shortcut_detected"] = SHORTCUT["p_holm"] < 0.05
    save_table(SHORTCUT.round(4), "shortcut_test", index=True)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.bar(
        SHORTCUT.index,
        SHORTCUT["background_only_accuracy"],
        color=np.where(SHORTCUT["shortcut_detected"], "tab:red", "tab:grey"),
    )
    ax.axhline(majority, color="black", linestyle="--", label="majority-class rate")
    ax.set_xticks(range(len(SHORTCUT)), SHORTCUT.index, rotation=30, ha="right")
    ax.set(
        title="Accuracy of models trained and tested on background-only images (red: above chance)",
        ylabel="Accuracy",
        ylim=(0, 1.02),
    )
    ax.legend()
    save_fig(fig, "17_shortcut_test")

## 8. Efficiency

Measured on one T4 after training, with nothing else on the device, under fp16 autocast: single-image latency (100 timed passes after 10 warm-up passes) and batch-32 throughput (20 passes after 3 warm-up), the whole protocol repeated three times; mean ± pooled standard deviation. Training cost comes from the workers: wall time per run, training images per second and peak training memory, averaged over folds. The two workers shared four CPU cores, so training throughput is a lower bound. Expected output: one table and an accuracy-latency figure.

In [ ]:
bench_rows = []
for name in tqdm(models_done, desc="Benchmark"):
    model, _ = load_trained(name)
    size = lib.MODEL_SPECS[name]["input_size"]
    single = torch.randn(1, 3, size, size, device=DEVICE)
    batch = torch.randn(THROUGHPUT_BATCH, 3, size, size, device=DEVICE)
    lat = (
        np.concatenate(
            [
                lib.time_passes(model, single, LATENCY_WARMUP, LATENCY_ITERS)
                for _ in range(BENCH_REPEATS)
            ]
        )
        * 1000
    )
    thr = np.concatenate(
        [
            THROUGHPUT_BATCH / lib.time_passes(model, batch, THROUGHPUT_WARMUP, THROUGHPUT_ITERS)
            for _ in range(BENCH_REPEATS)
        ]
    )
    runs = FULL[FULL["model"] == name]
    bench_rows.append(
        {
            "model": name,
            "parameters_m": PROFILES.loc[name, "parameters_m"],
            "gflops_native": PROFILES.loc[name, "gflops_native"],
            "gflops_224": PROFILES.loc[name, "gflops_224"],
            "latency_ms_mean": lat.mean(),
            "latency_ms_sd": lat.std(),
            "throughput_img_s_mean": thr.mean(),
            "throughput_img_s_sd": thr.std(),
            "train_minutes_per_run": runs["train_seconds"].mean() / 60,
            "train_img_s": runs["train_images_per_second"].mean(),
            "peak_train_memory_gb": runs["peak_train_memory_gb"].max(),
            "micro_batch": int(runs["micro_batch"].iloc[0]),
            "optimizer_steps": int(runs["optimizer_steps"].iloc[0]),
            "accuracy": LEADERBOARD.loc[name, "accuracy"],
            "accuracy_per_m_params": 100
            * LEADERBOARD.loc[name, "accuracy"]
            / PROFILES.loc[name, "parameters_m"],
        }
    )
    del model
    torch.cuda.empty_cache()
EFFICIENCY = pd.DataFrame(bench_rows).set_index("model")
save_table(EFFICIENCY.round(3), "efficiency", index=True)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    EFFICIENCY["latency_ms_mean"],
    EFFICIENCY["accuracy"],
    s=20 + 3 * EFFICIENCY["parameters_m"],
    alpha=0.6,
)
ax.errorbar(
    EFFICIENCY["latency_ms_mean"],
    EFFICIENCY["accuracy"],
    xerr=EFFICIENCY["latency_ms_sd"],
    fmt="none",
    ecolor="grey",
)
for name, row in EFFICIENCY.iterrows():
    ax.annotate(
        name,
        (row["latency_ms_mean"], row["accuracy"]),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points",
    )
ax.set(
    title="Test accuracy vs single-image latency (bubble area: parameters)",
    xlabel="Latency (ms, batch 1, fp16)",
    ylabel="Fold-ensemble test accuracy",
)
save_fig(fig, "18_accuracy_vs_latency")

### External validation

Runs only when `EXTERNAL_FOLDERS` names an attached collection. Its images, minus any near-duplicate of the pool, are scored by every model's fold ensemble. Expected output: an external leaderboard, or a note that none was configured.

In [ ]:
EXTERNAL_INDEX = np.flatnonzero((frame["role"] == "external").to_numpy())
if not len(EXTERNAL_INDEX):
    log(
        "No external evaluation: EXTERNAL_FOLDERS is empty or every external image duplicated the pool."
    )
else:
    y_ext = LABEL_OF[EXTERNAL_INDEX]
    ext_rows = []
    for name in models_done:
        ids = FULL.loc[FULL["model"] == name, "run_id"]
        probs = np.mean(
            [
                lib.logits_to_probs(
                    RUN_PROBS[r]["external_logits"], FULL.set_index("run_id").loc[r, "temperature"]
                )
                for r in ids
            ],
            axis=0,
        )
        ext_rows.append({"model": name, **lib.classification_metrics(y_ext, probs)})
    save_table(
        pd.DataFrame(ext_rows).set_index("model").round(4), "external_leaderboard", index=True
    )

## 9. Export and summary

### Web-app checkpoint

The web app loads `EfficientNetB0.pth` into torchvision's EfficientNet-B0 with a replaced `classifier[1]` (see `Backend/neuroscan/inference/networks.py`) and applies the same brain crop used here. The fold with the lowest validation loss is exported with its temperature and threshold, and the file is checked by loading it into exactly that architecture and reproducing the stored test logits. Copy `weights/deploy/EfficientNetB0.pth` to `Backend/Classification/` to deploy it. Expected output: the export path and a passed check.

In [ ]:
DEPLOY = WEIGHTS / "deploy"
DEPLOY.mkdir(exist_ok=True)
if DEPLOY_MODEL in models_done:
    run = best_fold_run(DEPLOY_MODEL)
    source = WEIGHTS / lib.slug(DEPLOY_MODEL) / f"{run['run_id']}.pth"
    target = DEPLOY / f"{DEPLOY_MODEL}.pth"
    shutil.copyfile(source, target)
    backend_model = torchvision.models.efficientnet_b0(weights=None)
    backend_model.classifier[1] = nn.Linear(backend_model.classifier[1].in_features, NUM_OUTPUTS)
    backend_model.load_state_dict(
        torch.load(target, map_location="cpu", weights_only=True), strict=True
    )
    backend_model = backend_model.to(DEVICE).eval()
    check = np.ascontiguousarray(test_images[:16])
    reproduced = lib.predict_logits(backend_model, lib.GpuPreprocessor(224, DEVICE), check)
    stored = np.load(PREDICTIONS / f"{run['run_id']}.npz")["test_logits"][:16]
    same_decision = np.array_equal(
        lib.logits_to_probs(reproduced).argmax(1), lib.logits_to_probs(stored).argmax(1)
    )
    max_diff = float(np.abs(reproduced - stored).max())
    assert (
        same_decision and max_diff < 0.1
    ), f"Exported checkpoint does not reproduce the stored logits (max diff {max_diff:.4f})"
    metadata = {
        "task": TASK,
        "architecture": "torchvision efficientnet_b0, classifier[1] replaced",
        "classes": CLASS_NAMES,
        "outputs": NUM_OUTPUTS,
        "input_size": 224,
        "preprocessing": "crop_to_brain (blur 5x5, threshold 45, largest contour box), resize, ImageNet normalisation",
        "run_id": run["run_id"],
        "temperature": float(run["temperature"]),
        "decision_threshold": (
            None
            if run["threshold"] is None or pd.isna(run["threshold"])
            else float(run["threshold"])
        ),
        "fold_test_accuracy": float(run["test_accuracy"]),
    }
    (DEPLOY / "metadata.json").write_text(json.dumps(metadata, indent=2))
    log(f"Exported {target} ({target.stat().st_size / 1e6:.1f} MB); reload check passed")
    del backend_model
else:
    log(f"{DEPLOY_MODEL} has no finished run; nothing exported")

### Summary

The headline table for the write-up, the run environment, and the limitations this notebook cannot remove. Expected output: the summary table and `metrics/run_summary.json`.

In [ ]:
summary_table = LEADERBOARD[
    [
        "family",
        "folds",
        "accuracy",
        "accuracy_ci",
        "balanced_accuracy",
        "f1_macro",
        "roc_auc",
        "ece",
    ]
].join(
    EFFICIENCY[
        [
            "parameters_m",
            "gflops_native",
            "latency_ms_mean",
            "latency_ms_sd",
            "throughput_img_s_mean",
        ]
    ]
)
if BINARY:
    summary_table = summary_table.join(
        LEADERBOARD[["sensitivity", "specificity", "tuned_sensitivity", "tuned_specificity"]]
    )
summary_table = summary_table.join(
    pd.DataFrame(diff_rows).set_index("model")[["separable_from_leader"]]
)
save_table(summary_table.round(4), "summary_table", index=True)
run_summary = {
    "task": TASK,
    "images_in_pool": int(len(POOL)),
    "images_after_deduplication": int(len(kept_index)),
    "label_noise_excluded": int(label_noise.sum()),
    "test_images": int(len(TEST_INDEX)),
    "folds": N_FOLDS,
    "jobs_finished": int(finished_jobs()),
    "jobs_total": len(JOBS),
    "leader": leader,
    "training_wall_minutes": TRAINING_WALL_SECONDS / 60,
    "notebook_minutes": (time.time() - NOTEBOOK_START) / 60,
    "environment": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "timm": timm.__version__,
        "cuda": torch.version.cuda,
        "gpus": [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)],
    },
    "limitations": [
        "MRI sequence of the images is not recorded by the sources",
        "binary negatives are tumour-negative, not certified normal",
        "no patient identifiers: grouping by image similarity is a proxy for a patient-level split",
        "two-dimensional public slices only; no institutional or three-dimensional external data",
    ],
}
save_json(run_summary, "run_summary")
log(
    f"Leader: {leader}; {run_summary['jobs_finished']}/{run_summary['jobs_total']} jobs; notebook time {run_summary['notebook_minutes']:.0f} min"
)

### Package the outputs

Zips everything in `/kaggle/working/` except the image cache and job claims, storing already-compressed files (weights, images) without recompression, and shows a download link. The zip also appears in the notebook's **Output** tab after a saved run. Expected output: the file count and a download link.

In [ ]:
ZIP_PATH = WORK / "outputs.zip"
EXCLUDE_PARTS = {"__pycache__", ".cache", ".ipynb_checkpoints", "cache", "tmp", "wandb"}
STORED_SUFFIXES = {
    ".zip",
    ".png",
    ".jpg",
    ".jpeg",
    ".pt",
    ".pth",
    ".bin",
    ".safetensors",
    ".gz",
    ".npz",
    ".npy",
}
files = [
    path
    for path in WORK.rglob("*")
    if path.is_file()
    and path != ZIP_PATH
    and not EXCLUDE_PARTS.intersection(path.relative_to(WORK).parts)
]
total_bytes = sum(path.stat().st_size for path in files)
free_bytes = shutil.disk_usage(WORK).free
assert (
    free_bytes > total_bytes * 1.05
), f"Not enough disk space to zip {total_bytes / 1e9:.2f} GB with {free_bytes / 1e9:.2f} GB free."
with zipfile.ZipFile(ZIP_PATH, "w", allowZip64=True) as archive:
    for path in tqdm(files, desc="Zipping outputs", unit="file"):
        method = (
            zipfile.ZIP_STORED if path.suffix.lower() in STORED_SUFFIXES else zipfile.ZIP_DEFLATED
        )
        archive.write(path, path.relative_to(WORK), compress_type=method)
print(f"Zipped {len(files)} files ({total_bytes / 1e9:.2f} GB) into {ZIP_PATH}", flush=True)
display(FileLink(os.path.relpath(ZIP_PATH)))